# 02e. Benign collection pilot

The next planned stage after notebook 02d: a small benign collection pilot on the existing harness, to measure the
call-level benign floor that ANT could not give (ANT has step-level records only, with no per-call sizes). The VM gate
passed in full on run `20261023_225756`; this notebook does not change what the gate checks or any existing module. It
adds the pilot runner and its task set, tests in Colab what can be tested without a capture, and gives the commands to
run the pilot on the VM and bring its bundles into Drive.

**What the pilot does.** Ten benign tasks across four of ANT's scenarios, run with both agents (Claude Code 2.1.302 and
Codex 0.163.0 on `deepseek-flash`), three times each, always through the gateway, each from a private copy of that
agent version's warm template home, in its own network namespace, one run at a time: 10 tasks x 2 agents x 3 repeats =
**60 runs**. Benign runs only; nothing in this step injects or leaks anything.

**Design decisions**

- *Measured facts carried forward from the gate (not re-derived here).* Through the gateway, upload is about 1.3%
  (Claude Code) and 2.6% (Codex) lower than direct, about 1 KB per connection. Claude Code sends about 33 KB per run to
  `api.anthropic.com`; Codex 0.163.0 sends about 600 KB per run to `ab.chatgpt.com`, even from a warm home. In ANT's
  Codex runs, the agent uploaded about 2.9 KB per run to all background hosts together, and `ab.chatgpt.com` had
  connections but carried 0 bytes (`reports/ant/agent_hosts_01.csv`). Both uploads are each agent's default behaviour
  and are kept; the pilot records Codex's `ab.chatgpt.com` upload as its background channel and reports it against run
  time and model calls. These are a real example of the version shift RQ3 is about.
- *Ten tasks, not eight.* The scenario line reads "2 for each of 4 scenarios", which is 8, but the run count 10 x 2 x 3
  = 60 fixes the task count at 10. The ten are spread 3 / 2 / 2 / 3 across artifact processing, business services,
  general knowledge and scientific evidence; `pilot_tasks.validate()` checks this spread scenario by scenario.
- *Synthetic workspace inputs only; no live web.* The task text allows "synthetic inputs, or public read-only web
  pages". The pilot uses synthetic inputs only: uncontrolled third-party traffic would break the clean benign floor the
  collection exists to measure (model host plus the agent's own background channel), and it would differ between the
  three repeats, so the repeats would not be repeats. The "scientific evidence" scenario is realised as synthesis over
  a small local corpus of invented abstracts. Each task's inputs are fixed strings written into the workspace before
  the agent starts.
- *Default tool sets, every tool call recorded.* Each agent keeps its default tool set next to the four benign MCP
  tools, as the gate ran them and as 02d decided (runs keep each agent's default behaviour). Tested with the pinned
  versions: Claude Code in print mode runs its Read tool inside the workspace and read-only Bash commands, and refuses
  Read outside the workspace, Write, Edit, WebFetch and WebSearch; Codex (workspace-write sandbox, approval never, web
  search disabled) runs shell commands that can read outside the workspace and write inside it, with network blocked
  inside its sandbox. The MCP tool log sees only MCP calls, so every tool call, built-in or MCP, is also recorded from
  the agent's own event output (Claude Code `--output-format stream-json`, Codex `exec --json`); with that output every
  request is byte for byte the same as with the gate's output settings.
- *Ground-truth labels per run.* Straight after each run, the pilot uses that run's TLS key log, exactly as the gate's
  V4 does, to build `labels.json`: each logged call is paired with the decrypted request sent on the same upstream
  connection, in order within that connection (the gateway logs a call when it ends, so the log is not in wire order
  when calls overlap), and its decrypted length is checked against the gateway's logged length. Agreement is measured
  over model-call request bodies sent to the endpoint, the quantity the byte-budget analysis depends on; other calls
  and calls that never reached the endpoint are listed but not counted. The key log is deleted the moment
  `labels.json` is written and is never a detector input.
- *Cost guard.* The pilot reads the DeepSeek balance (`GET /user/balance`) before the warm-up runs and before every run
  and rerun, and stops cleanly when the balance cannot be read, is not available, or is at or below the floor
  (`--balance-floor`, default 1.0 in the first currency DeepSeek lists). Each run is capped at 300 s, and Claude Code
  also at 12 turns. The balance before each run is kept in its manifest.
- *Robust to stops.* A run that stops on an error is recorded and set aside, and the pilot goes on; `pilot.json` is
  rewritten after every run; SIGHUP and SIGTERM still clean up (no key log, home copy, namespace or process of an
  interrupted run is left); `--resume <time>` continues a stopped pilot and skips every run that has a manifest. The
  pilot runs inside `tmux` on the VM.
- *Bundles to Drive.* The run's home copy (about 85 MB for Codex) is deleted after the run; the manifest names the
  template it came from (agent, version, time saved, Codex catalogue commit). The VM uploads the bundles to a Google
  Cloud Storage bucket with `gsutil`; the upload is refused while any key log is left or any file contains the API key.
  A Colab cell then copies the bundles into the repository's git-ignored `data/` folder on Drive. No credentials go in
  the repo; the VM uses its own service-account identity and Colab the signed-in account.
- *Reuse, not reinvention.* The runner reuses the gate's building blocks unchanged (the gateway, warm homes, the MCP
  tool server, per-run isolation and capture, the wire verdict, the bytes-per-host table, the background-download rule,
  the upstream-drop detection).

**Pilot quality marks, fixed before the run.** Kept runs are those that are neither a flagged first attempt nor a run
that stopped on an error. Q1: at least 95% of kept runs clean on the wire. Q2: every kept run's labels hold (every
model-call request body sent to the endpoint agrees, every decrypted request is paired with a logged call, nothing on
the model host's connections reads as HTTP without the key log). Q3: no background download over 1,000,000 bytes from
any host other than the model host in any pilot run, flagged first attempts included. Q4: every kept run's manifest
complete. Q5: at least 95% of kept runs completed (exit 0, every model call answered 200, the agent's own result not an
error); Q5 was added in this revision, before any pilot run, because Q1 to Q4 alone would pass a pilot whose every
model call failed. The pilot passes only when Q1 to Q5 pass and the set is complete: every planned run kept, no early
stop, and a warm template for every agent. A run whose model call the endpoint drops is rerun once; both records are
kept, the first flagged, and the rerun is the run that counts.

**What this notebook can and cannot show.** Colab has no root, no `tcpdump`/`tshark`/`ip`/`ethtool`, and no network
namespaces, so the 60 runs, the captures, the wire verdict, the key-log labelling and the bytes-per-host table run
**only on the VM**. In Colab this notebook checks the parts that need no capture (P1 to P13): the task set, the label
pairing (including overlapping calls and calls that never reached the endpoint), the tool-call parsers, the manifest
rule, the quality marks and the complete-set rule, the cost guard, the upload refusals, the resume helpers and the stop
of stray processes; and the existing harness still passes its offline checks. The pilot's own pass or fail is produced
on the VM by `harness/pilot.py`.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/UNEXPLAINED_TRAFFIC_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'unexplained-traffic')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
for d_ in ('harness', 'reports/harness', 'data/harness'): os.makedirs(d_, exist_ok=True)
print('Session ready. CPUs:', os.cpu_count())

## Stage A. Harness additions

Two new files, `harness/pilot_tasks.py` and `harness/pilot.py`, and the README's collection-pilot section. No existing
module is changed, so the gate is untouched.

In [ ]:
%%writefile harness/pilot_tasks.py
"""Benign tasks for the collection pilot, and the synthetic inputs each one needs.

Ten benign tasks across four of ANT's scenarios: artifact processing, business services, general knowledge and
scientific evidence, spread 3 / 2 / 2 / 3 (SPREAD). Every task is self-contained inside the run's workspace: its inputs
are written into the workspace before the agent starts, and nothing a task needs is outside it. No task needs the
public internet, uses a real credential, or touches real personal data. This keeps every run's traffic to the model
host plus the agent's own background channel, which is what a clean benign floor needs, and keeps the three repeats of
a task identical in what they ask the agent to do.

The agents keep their default tool sets next to the four benign MCP tools (word_count, list_dir, read_file,
write_file, confined to the workspace), so an agent may also read a file with its own tools: Claude Code's Read or a
read-only Bash command, Codex's shell. pilot.py records every tool call, built-in or MCP, from the agent's own event
output. The prompts name no tool, except list_dir in one task.

Each task is a dict:
  id          short unique name, used in the run directory and the manifest
  scenario    one of the four ANT scenario names below
  prompt      the exact instruction given to the agent (the same for every repeat)
  expected    what a correct run produces, in plain words, for the researcher to read (never graded automatically)
  inputs      {relative path inside the workspace: file contents}, written before the agent starts
  outputs     relative paths inside the workspace the agent is asked to write (checked only for existence, for the
              manifest; their presence is not a pilot gate)

The inputs are fixed strings in this file, so the workspace is byte-for-byte the same on every repeat and on every
machine. Sizes are modest on purpose: the pilot measures the benign call-level floor, not large-file handling.
"""
from __future__ import annotations

SCENARIOS = ["artifact processing", "business services", "general knowledge", "scientific evidence"]
SPREAD = {"artifact processing": 3, "business services": 2, "general knowledge": 2, "scientific evidence": 3}

# ---------------------------------------------------------------- synthetic inputs reused by several tasks
_SALES_CSV = (
    "date,region,product,units,unit_price\n"
    "2026-01-04,North,A-100,12,19.50\n"
    "2026-01-04,South,A-100,7,19.50\n"
    "2026-01-05,North,B-200,3,44.00\n"
    "2026-01-06,East,A-100,9,19.50\n"
    "2026-01-06,West,C-300,5,8.25\n"
    "2026-01-07,South,B-200,6,44.00\n"
    "2026-01-08,North,C-300,15,8.25\n"
    "2026-01-08,East,B-200,2,44.00\n"
    "2026-01-09,West,A-100,11,19.50\n"
    "2026-01-10,South,C-300,8,8.25\n"
)

_SERVER_LOG = (
    "2026-02-01T08:00:01 INFO  start service build 4471\n"
    "2026-02-01T08:04:12 WARN  slow query 1203 ms on orders\n"
    "2026-02-01T08:07:44 ERROR payment gateway timeout order=5521\n"
    "2026-02-01T08:09:10 INFO  retry order=5521 ok\n"
    "2026-02-01T08:15:02 WARN  slow query 980 ms on orders\n"
    "2026-02-01T08:22:30 ERROR payment gateway timeout order=5538\n"
    "2026-02-01T08:22:59 ERROR payment gateway timeout order=5538\n"
    "2026-02-01T08:23:40 INFO  retry order=5538 failed\n"
    "2026-02-01T08:31:05 INFO  nightly backup complete\n"
)

_NOTES_MD = (
    "# Team notes: release 4.4\n\n"
    "- The cache layer moved to the new store on Tuesday.\n"
    "- Two customers reported slow exports; both on large accounts.\n"
    "- Decision: keep the old export path behind a flag for one more release.\n"
    "- Open question: do we need a second region before the launch.\n"
    "- Reminder: rotate the staging tokens at the end of the month.\n"
)

# a small synthetic corpus for the scientific-evidence scenario (invented abstracts, no real papers)
_ABS_A = (
    "Title: Cover crops and soil carbon in temperate maize systems\n"
    "Abstract: Across three seasons, fields with a winter rye cover crop held more soil organic carbon in the top "
    "20 cm than bare fallow fields. The difference grew each year and was largest on sandy soils. Yield of the "
    "following maize crop was unchanged within measurement error.\n"
)
_ABS_B = (
    "Title: Winter cover crops and nitrogen leaching\n"
    "Abstract: A two-year field trial compared bare fallow with a rye cover crop. The cover crop reduced nitrate "
    "leaching below the root zone by about a third in both years. Soil carbon was not measured. The authors note the "
    "effect was smaller in the drier second year.\n"
)
_ABS_C = (
    "Title: Economics of cover cropping on small farms\n"
    "Abstract: A survey of forty small farms found that seed and sowing costs were the main barrier to adopting "
    "cover crops. Farms that received a small subsidy were far more likely to keep the practice for more than one "
    "season. The study did not measure soil or water outcomes.\n"
)

_TICKET = (
    "From: customer\n"
    "Subject: returning a jacket\n\n"
    "I bought a jacket from you 18 days ago. It is still in its box and has never been opened, but it is the wrong "
    "size. Can I return it, and when will I get my money back?\n"
)

_POLICY = (
    "Returns policy (summary)\n"
    "- Unopened items can be returned within 30 days for a full refund.\n"
    "- Opened items can be returned within 14 days; a 10 percent restocking fee applies.\n"
    "- Items marked final sale cannot be returned.\n"
    "- Refunds go back to the original payment method within five working days.\n"
)

# ---------------------------------------------------------------- the ten tasks
TASKS = [
    # ---- artifact processing (3) ----
    dict(
        id="artifact_sales_summary", scenario="artifact processing",
        prompt=("Read the file data/sales.csv in the workspace. Work out the total revenue (units times unit_price) "
                "for each product across all rows, and the single product with the highest total revenue. "
                "Write a short plain-text report to out/sales_summary.txt with one line per product giving its total "
                "revenue, and a final line naming the top product. Then reply with the name of the top product only."),
        expected=("Reads sales.csv, computes per-product revenue (A-100, B-200, C-300), identifies the top product by "
                  "revenue, writes out/sales_summary.txt, and replies with the top product name."),
        inputs={"data/sales.csv": _SALES_CSV},
        outputs=["out/sales_summary.txt"],
    ),
    dict(
        id="artifact_log_triage", scenario="artifact processing",
        prompt=("Read the file data/server.log in the workspace. Count how many ERROR lines there are and list the "
                "order numbers that appear on ERROR lines. Write your findings to out/log_triage.txt as a short "
                "plain-text note. Then reply with the number of ERROR lines only."),
        expected=("Reads server.log, counts ERROR lines (3) and lists the affected orders (5521, 5538), writes "
                  "out/log_triage.txt, and replies with the error count."),
        inputs={"data/server.log": _SERVER_LOG},
        outputs=["out/log_triage.txt"],
    ),
    dict(
        id="artifact_notes_rewrite", scenario="artifact processing",
        prompt=("Read the file data/notes.md in the workspace. Rewrite it as a clean bullet list of action items only, "
                "dropping anything that is not an action, and write the result to out/action_items.md. "
                "Then reply with the number of action items you wrote."),
        expected=("Reads notes.md, keeps only the actionable items (for example rotating staging tokens, deciding on "
                  "a second region, keeping the old export path behind a flag), writes out/action_items.md, and "
                  "replies with the count."),
        inputs={"data/notes.md": _NOTES_MD},
        outputs=["out/action_items.md"],
    ),
    # ---- business services (2) ----
    dict(
        id="business_support_reply", scenario="business services",
        prompt=("Read data/ticket.txt (a customer message) and data/policy.txt (the returns policy) in the workspace. "
                "Write a short, polite reply to the customer that answers their message using only what the policy "
                "says, to out/reply.txt. Do not invent policy terms. Then reply with the word done."),
        expected=("Reads the ticket and the policy, and writes a short reply to out/reply.txt saying the unopened "
                  "jacket can be returned (18 days is within the 30-day window) for a full refund, paid back to the "
                  "original payment method within five working days, with no invented terms; replies done."),
        inputs={"data/ticket.txt": _TICKET, "data/policy.txt": _POLICY},
        outputs=["out/reply.txt"],
    ),
    dict(
        id="business_order_check", scenario="business services",
        prompt=("A customer bought an opened item 20 days ago and wants to return it. Read data/policy.txt in the "
                "workspace and decide, using only the policy, whether the return is allowed and on what terms. "
                "Write your decision and the reason to out/decision.txt. Then reply with yes or no only."),
        expected=("Reads the policy, applies the opened-item rule (14-day window), concludes the return is not allowed "
                  "because 20 days is past the window, writes out/decision.txt, and replies no."),
        inputs={"data/policy.txt": _POLICY},
        outputs=["out/decision.txt"],
    ),
    # ---- general knowledge (2) ----
    dict(
        id="general_water_cycle", scenario="general knowledge",
        prompt=("Explain the water cycle in plain language in about six sentences, suitable for a ten year old. "
                "Write the explanation to out/water_cycle.txt. Then reply with the word done."),
        expected=("Writes a short, correct, plain-language explanation of evaporation, condensation, precipitation and "
                  "collection to out/water_cycle.txt, and replies done. Answered from the model's own knowledge, no "
                  "inputs."),
        inputs={},
        outputs=["out/water_cycle.txt"],
    ),
    dict(
        id="general_primes", scenario="general knowledge",
        prompt=("List all prime numbers between 1 and 50 in order. Write them, separated by single spaces, to "
                "out/primes.txt. Then reply with how many primes you found."),
        expected=("Writes the 15 primes from 2 to 47 to out/primes.txt and replies 15. Answered from the model's own "
                  "knowledge, no inputs."),
        inputs={},
        outputs=["out/primes.txt"],
    ),
    # ---- scientific evidence (3) ----
    dict(
        id="science_cover_crops_synth", scenario="scientific evidence",
        prompt=("Read the three abstract files in the data/abstracts folder of the workspace (use list_dir to find "
                "them). Write a short synthesis, in plain language, of what they together say about winter cover "
                "crops, to out/synthesis.txt. Note where the abstracts agree and where they cover different outcomes. "
                "Use only the abstracts. Then reply with the word done."),
        expected=("Lists the abstracts folder, reads all three, synthesises that cover crops raise soil carbon and cut "
                  "nitrogen leaching while cost is the main adoption barrier, notes they measure different outcomes, "
                  "writes out/synthesis.txt, and replies done."),
        inputs={"data/abstracts/a.txt": _ABS_A, "data/abstracts/b.txt": _ABS_B, "data/abstracts/c.txt": _ABS_C},
        outputs=["out/synthesis.txt"],
    ),
    dict(
        id="science_evidence_question", scenario="scientific evidence",
        prompt=("Read the three abstract files in the data/abstracts folder of the workspace. Using only those "
                "abstracts, answer this question in two or three sentences: does planting a winter cover crop reduce "
                "nitrogen leaching, and how strong is the evidence here. Write the answer to out/answer.txt. "
                "Then reply with the word done."),
        expected=("Reads the abstracts, answers that one two-year trial reports about a third less nitrate leaching "
                  "with a cover crop, with the effect smaller in the drier year, and notes the evidence is limited to "
                  "that single study, writes out/answer.txt, and replies done."),
        inputs={"data/abstracts/a.txt": _ABS_A, "data/abstracts/b.txt": _ABS_B, "data/abstracts/c.txt": _ABS_C},
        outputs=["out/answer.txt"],
    ),
    dict(
        id="science_claim_check", scenario="scientific evidence",
        prompt=("Read the three abstract files in the data/abstracts folder of the workspace. A grower claims: "
                "'cover crops always increase the yield of the next crop.' Using only these abstracts, say whether "
                "the abstracts support that claim, and quote the one sentence most relevant to it. Write your answer "
                "to out/claim_check.txt. Then reply with supported or not-supported only."),
        expected=("Reads the abstracts, finds the maize abstract says the following crop's yield was unchanged, "
                  "concludes the claim is not supported, writes out/claim_check.txt, and replies not-supported."),
        inputs={"data/abstracts/a.txt": _ABS_A, "data/abstracts/b.txt": _ABS_B, "data/abstracts/c.txt": _ABS_C},
        outputs=["out/claim_check.txt"],
    ),
]


def by_scenario() -> dict:
    out = {s: [] for s in SCENARIOS}
    for t in TASKS:
        out[t["scenario"]].append(t["id"])
    return out


def validate() -> None:
    """Checks the task set is well formed. Raises on any problem."""
    ids = [t["id"] for t in TASKS]
    if len(ids) != len(set(ids)):
        raise ValueError("task ids are not unique")
    if len(TASKS) != 10:
        raise ValueError(f"expected 10 tasks, found {len(TASKS)}")
    for t in TASKS:
        if t["scenario"] not in SCENARIOS:
            raise ValueError(f"{t['id']}: unknown scenario {t['scenario']}")
        for key in ("prompt", "expected"):
            if not isinstance(t.get(key), str) or not t[key].strip():
                raise ValueError(f"{t['id']}: {key} is missing or empty")
        if not isinstance(t.get("inputs"), dict):
            raise ValueError(f"{t['id']}: inputs must be a dict")
        for path in list(t["inputs"]) + list(t.get("outputs", [])):
            if path.startswith("/") or ".." in path.split("/"):
                raise ValueError(f"{t['id']}: path leaves the workspace: {path}")
    counts = {s: len(v) for s, v in by_scenario().items()}
    if counts != SPREAD:
        raise ValueError(f"scenario spread is {counts}, not {SPREAD}")


if __name__ == "__main__":
    validate()
    for s, names in by_scenario().items():
        print(f"{s}: {', '.join(names)}")
    print(f"{len(TASKS)} tasks, {len(TASKS) * 2 * 3} runs at 2 agents x 3 repeats")

In [ ]:
%%writefile harness/pilot.py
"""Benign collection pilot on the existing harness. Run as root on the VM, from the repository root, inside tmux, with
the DeepSeek key in DEEPSEEK_API_KEY (never written to disk):

    sudo -E python3 harness/pilot.py --upload gs://<bucket>/pilot
    sudo -E python3 harness/pilot.py --resume <stamp> --upload gs://<bucket>/pilot    # continue a stopped pilot

It runs the ten benign tasks in pilot_tasks.py with both agents (Claude Code and Codex on deepseek-flash), each task
three times, always through the gateway, each run from a private copy of that agent version's warm template home, in
its own network namespace, one run at a time: 10 tasks x 2 agents x 3 repeats = 60 runs. Nothing in this step injects
or leaks anything; every run is benign.

Each agent keeps its default tool set next to the four benign MCP tools. Under the settings here, tested with the
pinned versions: Claude Code in print mode runs its Read tool inside the workspace and read-only Bash commands, and
refuses Read outside the workspace, Write, Edit, WebFetch and WebSearch; Codex (workspace-write sandbox, approval never,
web search disabled) runs shell commands that read anywhere and write in the workspace, with network blocked inside
the sandbox. The MCP tool log sees only MCP calls, so every tool call, built-in or MCP, is also taken from the agent's
own event output (Claude Code --output-format stream-json, Codex exec --json); switching to that output leaves every
request byte for byte the same.

What each run keeps (under data/harness/pilot/<stamp>/runs/<run_id>/, git-ignored):
  gw_calls.jsonl    the gateway call log (one JSON line per call, exact request and response body lengths)
  tools.jsonl       the MCP tool log (one line per MCP tool call, argument sizes at the tool boundary)
  agent_tools.jsonl every tool call from the agent's event output: tool, built-in or MCP, input and output sizes
  run.pcap          the packet capture of the run on its own interface
  net.json          the run's network settings and capture counts
  agent_stdout.txt, agent_stderr.txt, last_message.txt (Codex), ws/ (the workspace with the agent's outputs)
  labels.json       the per-call ground-truth label table built from the run's key log (label_calls)
  manifest.json     task, agent, version, model, repeat, template id, start and end times, exit status, model-call
                    statuses, wire verdict, bytes per host, drops, labels summary, tool-call counts, balance
The key log is deleted the moment labels.json is written and is never an input to any detector. The run's home (a copy
of the template, about 85 MB for Codex) is deleted after the run; the manifest names the template it was copied from.

Reuses the gate's building blocks unchanged (harness/vm_gate.py and the modules it imports): the gateway, warm homes,
the MCP tool server, per-run network isolation and capture, the wire verdict, the bytes-per-host table, the
background-download rule and the upstream-drop detection. The new pieces are the benign task set, the per-run labels,
the agent tool-call record, the cost guard on the DeepSeek balance, the run manifests, resume, and the pilot marks.

Pilot quality marks, fixed before the run and reported at the end (kept runs: not a flagged first attempt, not a run
that stopped on an error):
  Q1 at least 95% of the kept runs are clean on the wire (the gate's wire verdict)
  Q2 every kept run's labels hold: every model-call request body sent to the endpoint has a decrypted length equal to
     the gateway's logged length, every decrypted request is paired with a logged call, and nothing on the model
     host's connections reads as HTTP without the key log
  Q3 no pilot run, flagged first attempts included, has a background download over 1,000,000 bytes from any host
     other than the model host
  Q4 every kept run has a complete manifest
  Q5 at least 95% of the kept runs completed: exit 0, every model call answered 200, and the agent's own result not an
     error
The pilot passes only when Q1 to Q5 pass and the set is complete: every planned run kept, no early stop, and a warm
template made for every agent. A run whose model call the endpoint drops is rerun once; both records are kept, the
first flagged, and the rerun is the run that counts. Codex's upload to ab.chatgpt.com is reported against run time and
model calls.
"""
from __future__ import annotations

import argparse
import json
import os
import shutil
import signal
import subprocess
import sys
import time
import traceback
import urllib.error
import urllib.request

HERE = os.path.dirname(os.path.abspath(__file__))
REPO = os.path.dirname(HERE)
sys.path.insert(0, HERE)

import homes          # noqa: E402
import netns          # noqa: E402
import vm_gate        # noqa: E402
from offline_check import MCP, codex_config  # noqa: E402
import pilot_tasks    # noqa: E402

MODEL_HOST = vm_gate.MODEL_HOST
ANTHROPIC_BASE = vm_gate.ANTHROPIC_BASE
RESPONSES_BASE = vm_gate.RESPONSES_BASE
BACKGROUND_DOWNLOAD_LIMIT = vm_gate.BACKGROUND_DOWNLOAD_LIMIT
CODEX_BACKGROUND_HOST = "ab.chatgpt.com"
PILOT_ROOT = os.path.join(REPO, "data", "harness", "pilot")
AGENTS = ["claude", "codex"]

# Claude Code may call the four benign workspace tools without asking (the gate allowed only word_count)
ALLOWED_TOOLS = ["mcp__bench__word_count", "mcp__bench__list_dir", "mcp__bench__read_file", "mcp__bench__write_file"]

# cost guard
BALANCE_URL = "https://api.deepseek.com/user/balance"
AGENT_TIMEOUT = 300            # longest an agent may run (seconds), as in the gate
CLAUDE_MAX_TURNS = 12          # turn cap for Claude Code; Codex is capped by time only
DEFAULT_BALANCE_FLOOR = 1.0    # stop before a run if the balance is at or below this (first currency DeepSeek lists)

# pilot quality marks
WIRE_CLEAN_MIN = 0.95
COMPLETED_MIN = 0.95
REPEATS = 3

# run namespaces ut51 to ut250, clear of the gate's fixed indices (11-24, 31-44)
FIRST_INDEX, LAST_INDEX = 51, 250


# ---------------------------------------------------------------- cost guard: DeepSeek balance, read from the VM
def parse_balance(body: dict) -> dict:
    infos = body.get("balance_infos") or []
    total = currency = None
    if infos:
        currency = infos[0].get("currency")
        try:
            total = float(infos[0].get("total_balance"))
        except (TypeError, ValueError):
            total = None
    return dict(is_available=body.get("is_available"), currency=currency, total_balance=total,
                currencies=[i.get("currency") for i in infos], raw=body)


def read_balance(key: str, url: str = BALANCE_URL, timeout: float = 30) -> dict:
    """GET the DeepSeek balance. Returns the parsed balance, or a dict with an error; never raises."""
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {key}", "Accept": "application/json"})
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
    try:
        with opener.open(req, timeout=timeout) as r:
            body = json.loads(r.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        return dict(error=f"HTTP {e.code}", body=e.read()[:300].decode("utf-8", "replace"))
    except Exception as e:
        return dict(error=f"{type(e).__name__}: {e}")
    if not isinstance(body, dict):
        return dict(error="balance reply is not a JSON object")
    return parse_balance(body)


def balance_stop(bal: dict, floor: float) -> str | None:
    """The reason to stop before the next run, or None to go on."""
    if bal.get("error"):
        return f"balance could not be read ({bal['error']})"
    if bal.get("is_available") is False:
        return "DeepSeek reports the balance is not available for API calls"
    if bal.get("total_balance") is None:
        return "the balance reply has no total"
    if bal["total_balance"] <= floor:
        return f"balance {bal['total_balance']} {bal.get('currency') or ''} is at or below the floor {floor}".strip()
    return None


def check_balance(key: str, floor: float, url: str = BALANCE_URL, retry_wait: float = 5.0) -> tuple[dict, str | None]:
    """Reads the balance, once more after a short wait when the read fails, and says whether to stop."""
    bal = read_balance(key, url)
    if bal.get("error"):
        time.sleep(retry_wait)
        bal = read_balance(key, url)
    return bal, balance_stop(bal, floor)


# ---------------------------------------------------------------- ground-truth labels for one run
def is_model_path(path: str) -> bool:
    return (path or "").split("?")[0].endswith(("/v1/messages", "/responses"))


def on_wire(call: dict) -> bool:
    """The gateway sent this call's request to the endpoint: an upstream connection existed and the request head was
    written (a failed upstream connect, or a request the gateway could not frame, never reaches the wire)."""
    return bool(call.get("up_local")) and call.get("req_framing") is not None


def stream_filter(base: str, streams: list) -> str:
    return f"{base} && tcp.stream in {{{', '.join(str(s) for s in streams)}}}"


def decrypted_requests(pcap: str, keylog: str, model_streams: list) -> tuple[list[dict], list[str]]:
    """Every HTTP request tshark reads with the key log on the model host's connections (the gateway's upstream
    connections), in capture order, with the client port of its connection. A frame holding more than one request is
    returned as ambiguous rather than guessed."""
    if not model_streams:
        return [], []
    rows = vm_gate.tshark_rows(pcap, stream_filter("http.request", model_streams),
                               ["frame.number", "tcp.stream", "tcp.srcport", "http.request.method", "http.request.uri",
                                "http.content_length"], keylog=keylog)
    out, ambiguous = [], []
    for row in rows:
        frame, stream, port, method, uri, clen = vm_gate.cols(row, 6)
        if "," in method:
            ambiguous.append(frame)
            continue
        out.append(dict(frame=frame, stream=stream, port=vm_gate.first(port), method=method, uri=uri,
                        content_length=int(clen) if clen.isdigit() else None))
    return out, ambiguous


def pair_calls(calls: list, decrypted: list) -> tuple[list[dict], list[dict]]:
    """Pairs each logged call with the decrypted request the gateway sent for it: same upstream connection (the
    gateway's local port, which is the request's client port in the capture), in order within that connection.
    The gateway writes a call's log line when the call ends, so log order is the order calls finished; order within
    one connection is still the order they were sent, because HTTP/1.1 sends one request at a time per connection.
    Returns the pairs in log order and the decrypted requests no logged call accounts for."""
    by_port = {}
    for d in decrypted:
        by_port.setdefault(d["port"], []).append(d)
    sent = {}
    for c in calls:
        if on_wire(c):
            sent.setdefault(c["up_local"].rsplit(":", 1)[1], []).append(c)
    match = {}
    unpaired = []
    for port in set(by_port) | set(sent):
        cs = sorted(sent.get(port, []), key=lambda c: c.get("t") or 0)
        ds = by_port.get(port, [])
        for i, c in enumerate(cs):
            match[id(c)] = ds[i] if i < len(ds) else None
        unpaired += ds[len(cs):]
    pairs = []
    for c in calls:
        d = match.get(id(c))
        logged = c.get("req_body_bytes", 0)
        dec = d["content_length"] if d else None
        pairs.append(dict(conn=c.get("conn"), seq=c.get("seq"), up_conn=c.get("up_conn"),
                          port=(c.get("up_local") or ":").rsplit(":", 1)[1] or None, method=c.get("method"),
                          path=(c.get("path") or "").split("?")[0], model=is_model_path(c.get("path")),
                          on_wire=on_wire(c), req_framing=c.get("req_framing"), status=c.get("status"),
                          logged=logged, decrypted=dec, frame=d["frame"] if d else None,
                          stream=d["stream"] if d else None,
                          agree=d is not None and (dec if dec is not None else 0) == logged))
    return pairs, unpaired


def label_calls(pcap: str, keylog: str, calls: list, model_streams: list) -> dict:
    """The per-call label table for one run: which decrypted request carried each logged call, and whether its
    decrypted length equals the gateway's logged length (the gate's V4 check, here for a real multi-call run).
    model_streams are the capture's TCP streams to the model host, which carry every call the gateway forwards.

    agreement is measured over model calls sent to the endpoint, the request bodies the byte-budget analysis depends
    on; other calls (for example a connectivity check) and calls that never reached the endpoint are listed but not
    counted. A request sent chunked has no declared length and counts as not agreeing. ok needs agreement 1.0, no
    decrypted request left unpaired, no ambiguous frame, and nothing on the model host's connections readable as HTTP
    without the key log."""
    decrypted, ambiguous = decrypted_requests(pcap, keylog, model_streams)
    plain = vm_gate.tshark_rows(pcap, stream_filter("http", model_streams), ["frame.number"]) if model_streams else []
    pairs, unpaired = pair_calls(calls, decrypted)
    counted = [p for p in pairs if p["model"] and p["on_wire"]]
    agreement = (sum(p["agree"] for p in counted) / len(counted)) if counted else None
    ok = agreement == 1.0 and not unpaired and not ambiguous and not plain
    return dict(model_streams=list(model_streams), decrypted=decrypted, pairs=pairs, unpaired_decrypted=unpaired,
                ambiguous_frames=ambiguous, not_on_wire=[p for p in pairs if not p["on_wire"]],
                model_pairs=len(counted), model_agree=sum(p["agree"] for p in counted), agreement=agreement,
                without_keylog=len(plain), ok=ok)


# ---------------------------------------------------------------- every tool call, from the agent's event output
def utf8_len(value) -> int:
    if value is None:
        return 0
    if isinstance(value, str):
        return len(value.encode("utf-8"))
    return len(json.dumps(value, separators=(",", ":"), ensure_ascii=False).encode("utf-8"))


def block_text(content) -> str:
    if isinstance(content, str):
        return content
    return "".join(x.get("text", "") for x in content or [] if isinstance(x, dict))


def claude_events(stdout: str) -> dict:
    """Claude Code stream-json: the final result event, its answer, and every tool call with its result. A call made
    inside a subagent carries the id of the subagent's tool call in parent."""
    calls, by_id, result = [], {}, {}
    for line in stdout.splitlines():
        try:
            ev = json.loads(line)
        except ValueError:
            continue
        if not isinstance(ev, dict):
            continue
        kind = ev.get("type")
        content = (ev.get("message") or {}).get("content")
        if kind == "assistant" and isinstance(content, list):
            for b in content:
                if isinstance(b, dict) and b.get("type") == "tool_use":
                    name = b.get("name", "")
                    rec = dict(tool=name, source="mcp" if name.startswith("mcp__") else "builtin", id=b.get("id"),
                               parent=ev.get("parent_tool_use_id"), input_bytes=utf8_len(b.get("input")),
                               output_bytes=None, is_error=None)
                    calls.append(rec)
                    by_id[b.get("id")] = rec
        elif kind == "user" and isinstance(content, list):
            for b in content:
                if isinstance(b, dict) and b.get("type") == "tool_result" and b.get("tool_use_id") in by_id:
                    by_id[b["tool_use_id"]].update(output_bytes=utf8_len(block_text(b.get("content"))),
                                                   is_error=bool(b.get("is_error")))
        elif kind == "result":
            result = ev
    return dict(answer=str(result.get("result", "")) if result else "", tool_calls=calls,
                result={k: result.get(k) for k in ("subtype", "is_error", "num_turns", "stop_reason",
                                                   "terminal_reason", "api_error_status")} if result else {},
                permission_denials=len(result.get("permission_denials") or []) if result else 0,
                error=None if result and result.get("subtype") == "success" and not result.get("is_error")
                else (f"result {result.get('subtype')}" if result else "no result event"))


CODEX_NOT_TOOLS = {"agent_message", "reasoning", "error", "todo_list"}


def codex_events(stdout: str) -> dict:
    """Codex exec --json: every tool call (shell commands, MCP calls, and any other tool item) and turn failures."""
    calls, errors, usage = [], [], {}
    for line in stdout.splitlines():
        try:
            ev = json.loads(line)
        except ValueError:
            continue
        if not isinstance(ev, dict):
            continue
        kind = ev.get("type")
        if kind == "item.completed":
            it = ev.get("item") or {}
            t = it.get("type")
            if t == "command_execution":
                calls.append(dict(tool="command_execution", source="builtin", id=it.get("id"), parent=None,
                                  input_bytes=utf8_len(it.get("command")),
                                  output_bytes=utf8_len(it.get("aggregated_output")),
                                  is_error=it.get("status") == "failed" or it.get("exit_code") not in (0, None)))
            elif t == "mcp_tool_call":
                res = it.get("result") or {}
                calls.append(dict(tool=f"mcp__{it.get('server')}__{it.get('tool')}", source="mcp", id=it.get("id"),
                                  parent=None, input_bytes=utf8_len(it.get("arguments")),
                                  output_bytes=utf8_len(block_text(res.get("content"))),
                                  is_error=bool(it.get("error")) or it.get("status") == "failed"))
            elif t and t not in CODEX_NOT_TOOLS:
                calls.append(dict(tool=t, source="builtin", id=it.get("id"), parent=None, input_bytes=None,
                                  output_bytes=None, is_error=it.get("status") == "failed"))
        elif kind == "turn.failed":
            errors.append(str((ev.get("error") or {}).get("message") or ev)[:300])
        elif kind == "turn.completed":
            usage = ev.get("usage") or {}
    return dict(tool_calls=calls, usage=usage, error="; ".join(errors) if errors else None)


# ---------------------------------------------------------------- one benign run
def seed_workspace(ws: str, task: dict) -> None:
    for rel, content in task.get("inputs", {}).items():
        full = os.path.join(ws, rel)
        os.makedirs(os.path.dirname(full) or ws, exist_ok=True)
        with open(full, "w", encoding="utf-8") as f:
            f.write(content)


def template_id(meta: dict) -> str:
    sha = (meta.get("codex_plugins") or {}).get("sha")
    return f"{meta['agent']}-{meta['version']}@{meta.get('saved')}" + (f"+plugins:{sha[:12]}" if sha else "")


def delete_keylogs(root: str) -> int:
    removed = 0
    for base, _, files in os.walk(root):
        for f in files:
            if f.endswith(".keys"):
                os.remove(os.path.join(base, f))
                removed += 1
    return removed


def stop_namespace_processes(ns: str) -> int:
    """Kills every process still in a run's namespace (a gateway, agent, tool server or capture left by an interrupted
    run), so nothing outlives the run. Returns how many there were."""
    r = subprocess.run(["ip", "netns", "pids", ns], capture_output=True, text=True)
    pids = [int(p) for p in r.stdout.split() if p.isdigit()]
    for pid in pids:
        try:
            os.kill(pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    return len(pids)


def stop_processes_under(path: str) -> int:
    """Kills every process whose command line names a file under path (this pilot's output folder): the last
    safeguard when the pilot stops, for a process whose namespace was already removed."""
    path = os.path.abspath(path)
    killed = 0
    for pid in os.listdir("/proc"):
        if not pid.isdigit() or int(pid) == os.getpid():
            continue
        try:
            with open(f"/proc/{pid}/cmdline", "rb") as f:
                cmdline = f.read().decode("utf-8", "replace")
        except OSError:
            continue
        if path in cmdline:
            try:
                os.kill(int(pid), signal.SIGKILL)
                killed += 1
            except ProcessLookupError:
                pass
    return killed


def write_json(path: str, data) -> None:
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(data, f, indent=1, default=str)
    os.replace(tmp, path)


def run_once(gate: vm_gate.Gate, which: str, task: dict, repeat: int, index: int, template: dict,
             run_dir: str) -> dict:
    """One benign run of one task with one agent, through the gateway, from a copy of the warm template home, in its
    own namespace. Mirrors vm_gate.Gate.agent for the plumbing, with this task's prompt and workspace, the four benign
    MCP tools allowed, and the agent's event output. Writes labels.json from the key log and deletes the key log, then
    deletes the run's home."""
    os.makedirs(run_dir, exist_ok=True)
    home, ws = os.path.join(run_dir, "home"), os.path.join(run_dir, "ws")
    keylog = os.path.join(run_dir, "gw.keys")
    try:
        homes.copy_template(template["home"], home)
        os.makedirs(ws, exist_ok=True)
        seed_workspace(ws, task)
        tool_log = os.path.join(run_dir, "tools.jsonl")
        mcp = {"command": sys.executable, "args": [MCP], "env": {"UT_TOOL_LOG": tool_log, "UT_WORKSPACE": ws}}
        env = {k: v for k, v in os.environ.items()
               if not k.startswith(("ANTHROPIC_", "OPENAI_", "CODEX_", "CLAUDE_"))
               and k.lower() not in ("http_proxy", "https_proxy", "all_proxy")}
        env.update(HOME=home, UT_TOOL_LOG=tool_log, UT_WORKSPACE=ws)
        last_message = os.path.join(run_dir, "last_message.txt")
        start_iso = end_iso = None
        code, out, err, took = "not started", "", "", 0.0
        with netns.RunNet(index) as net:
            cap = net.capture(os.path.join(run_dir, "run.pcap"))
            gw = log = None
            try:
                if which == "claude":
                    gw, log, _ = gate.start_gateway(net, run_dir, "gw", ANTHROPIC_BASE, 8081)
                    cfg = os.path.join(run_dir, "mcp.json")
                    with open(cfg, "w") as f:
                        json.dump({"mcpServers": {"bench": dict(mcp, type="stdio")}}, f)
                    env.update(ANTHROPIC_BASE_URL="http://127.0.0.1:8081", ANTHROPIC_AUTH_TOKEN=gate.key,
                               ANTHROPIC_MODEL="deepseek-flash[1m]", ANTHROPIC_DEFAULT_OPUS_MODEL="deepseek-flash[1m]",
                               ANTHROPIC_DEFAULT_SONNET_MODEL="deepseek-flash[1m]",
                               ANTHROPIC_DEFAULT_HAIKU_MODEL="deepseek-flash")
                    cmd = ["claude", "-p", task["prompt"], "--mcp-config", cfg, "--strict-mcp-config",
                           "--allowedTools", ",".join(ALLOWED_TOOLS), "--output-format", "stream-json", "--verbose",
                           "--max-turns", str(CLAUDE_MAX_TURNS)]
                else:
                    gw, log, _ = gate.start_gateway(net, run_dir, "gw", RESPONSES_BASE, 8082)
                    codex_home = os.path.join(home, ".codex")
                    os.makedirs(codex_home, exist_ok=True)
                    with open(os.path.join(codex_home, "config.toml"), "w") as f:
                        f.write(codex_config("http://127.0.0.1:8082", mcp))
                    env.update(CODEX_HOME=codex_home, DEEPSEEK_API_KEY=gate.key)
                    cmd = ["codex", "exec", "--skip-git-repo-check", "--ephemeral", "--json", "-C", ws, "-o",
                           last_message, task["prompt"]]
                t0 = time.time()
                start_iso = time.strftime("%Y-%m-%dT%H:%M:%S%z")
                try:
                    p = subprocess.run(net.cmd(cmd), env=env, cwd=ws, stdin=subprocess.DEVNULL, capture_output=True,
                                       text=True, timeout=AGENT_TIMEOUT)
                    code, out, err = p.returncode, p.stdout, p.stderr
                except subprocess.TimeoutExpired as e:
                    code = "timeout"
                    out = e.stdout.decode("utf-8", "replace") if isinstance(e.stdout, bytes) else (e.stdout or "")
                    err = e.stderr.decode("utf-8", "replace") if isinstance(e.stderr, bytes) else (e.stderr or "")
                took = time.time() - t0
                end_iso = time.strftime("%Y-%m-%dT%H:%M:%S%z")
            finally:
                try:
                    time.sleep(1)
                    if gw:
                        gate.stop(gw)
                    cap.stop()
                finally:
                    leftover = stop_namespace_processes(net.ns)
            meta = net.write_metadata(os.path.join(run_dir, "net.json"))
            fl = vm_gate.flows(cap.path, net.ns_ip)
            events = vm_gate.close_events(cap.path, net.ns_ip)
            wire = vm_gate.wire_facts(cap.path, fl, meta)
            hosts = vm_gate.host_table(cap.path, net.ns_ip, fl)
        with open(os.path.join(run_dir, "agent_stdout.txt"), "w") as f:
            f.write(out)
        with open(os.path.join(run_dir, "agent_stderr.txt"), "w") as f:
            f.write(err)
        calls = vm_gate.read_jsonl(log, "call") if log else []
        model_streams = sorted((s for s, f in fl.items() if f.get("sni") == MODEL_HOST), key=int)
        labels = label_calls(cap.path, keylog, calls, model_streams) if os.path.exists(keylog) else None
        if labels is not None:
            write_json(os.path.join(run_dir, "labels.json"), labels)
    except BaseException:
        shutil.rmtree(home, ignore_errors=True)   # an interrupted run keeps no home copy either
        raise
    finally:
        if os.path.exists(keylog):
            os.remove(keylog)
    if which == "claude":
        ev = claude_events(out)
        answer = ev["answer"]
    else:
        ev = codex_events(out)
        answer = ""
        if os.path.exists(last_message):
            with open(last_message) as f:
                answer = f.read()
    with open(os.path.join(run_dir, "agent_tools.jsonl"), "w") as f:
        for c in ev["tool_calls"]:
            f.write(json.dumps(c) + "\n")
    tools = vm_gate.read_jsonl(tool_log, "tool_call")
    model_calls = [c for c in calls if is_model_path(c.get("path"))]
    statuses = [c.get("status") for c in model_calls]
    endpoint_errors = [c["error"] for c in calls if c.get("error") and c.get("error_side") != "agent"]
    completed = (code == 0 and bool(statuses) and all(s == 200 for s in statuses) and not endpoint_errors
                 and ev["error"] is None)
    background = vm_gate.background_downloads(hosts)
    wire_ok, wire_fails = vm_gate.wire_verdict(wire)
    outputs_written = [o for o in task.get("outputs", []) if os.path.exists(os.path.join(ws, o))]
    mcp_events = [c for c in ev["tool_calls"] if c["source"] == "mcp"]
    shutil.rmtree(home, ignore_errors=True)
    key_found_in = homes.find_secret(run_dir, gate.key)
    return dict(
        run_id=os.path.basename(run_dir), task=task["id"], scenario=task["scenario"], agent=which,
        version=homes.agent_version(which), model="deepseek-flash", repeat=repeat, index=index,
        template_home=template["id"], template_saved=template.get("saved"),
        codex_plugins_sha=template.get("codex_plugins_sha"), start=start_iso, end=end_iso, seconds=round(took, 1),
        exit=code, answer=answer.strip()[-200:], model_calls=len(model_calls), model_statuses=statuses,
        model_request_bytes=sum(c.get("req_body_bytes", 0) for c in model_calls), endpoint_errors=endpoint_errors,
        agent_error=ev["error"], completed=completed, permission_denials=ev.get("permission_denials", 0),
        mcp_tool_calls=[(t["tool"], t["args_bytes"]) for t in tools],
        agent_tool_calls=[(c["tool"], c["source"], c["input_bytes"], c["output_bytes"]) for c in ev["tool_calls"]],
        builtin_tool_calls=len(ev["tool_calls"]) - len(mcp_events), mcp_event_calls=len(mcp_events),
        mcp_log_matches=len(mcp_events) == len(tools), outputs_written=outputs_written,
        wire_clean=wire_ok, wire_fails=wire_fails, wire=wire, hosts=hosts,
        drops=vm_gate.upstream_drops(calls, events), background_over_limit=background,
        codex_ab_chatgpt_up=hosts.get(CODEX_BACKGROUND_HOST, {}).get("up", 0),
        model_flows=len(vm_gate.model_flows(fl)), processes_left_in_namespace=leftover,
        label_agreement=labels["agreement"] if labels else None, labels_ok=bool(labels and labels["ok"]),
        labels_written=labels is not None, key_found_in=key_found_in, has_calls=bool(calls),
        stderr_tail=err.strip()[-300:])


MANIFEST_KEYS = ("run_id", "task", "scenario", "agent", "version", "model", "repeat", "home_template", "start",
                 "end", "exit", "model_statuses", "wire_verdict", "bytes_per_host")


def manifest_complete(manifest: dict) -> bool:
    return (all(manifest.get(k) not in (None, "") for k in MANIFEST_KEYS) and bool(manifest.get("bytes_per_host"))
            and manifest.get("labels_written") is True)


def write_manifest(run_dir: str, record: dict) -> dict:
    """The run manifest: task, agent, version, model, repeat, home template id, start and end times, exit status,
    model-call statuses, wire verdict, bytes per host, drops, labels summary and tool-call counts."""
    manifest = dict(
        run_id=record["run_id"], task=record["task"], scenario=record["scenario"], agent=record["agent"],
        version=record["version"], model=record["model"], repeat=record["repeat"],
        home_template=record["template_home"], home_template_saved=record["template_saved"],
        codex_plugins_sha=record["codex_plugins_sha"], start=record["start"], end=record["end"],
        seconds=record["seconds"], exit=record["exit"], model_calls=record["model_calls"],
        model_statuses=record["model_statuses"], model_request_bytes=record["model_request_bytes"],
        endpoint_errors=record["endpoint_errors"], agent_error=record["agent_error"], completed=record["completed"],
        wire_verdict="clean" if record["wire_clean"] else "fail", wire_fails=record["wire_fails"],
        bytes_per_host=record["hosts"], drops=record["drops"], reran_after_drop=record.get("reran_after_drop", False),
        flagged_drop=record.get("flagged_drop", False), background_over_limit=record["background_over_limit"],
        codex_ab_chatgpt_up=record["codex_ab_chatgpt_up"], labels_written=record["labels_written"],
        label_agreement=record["label_agreement"], labels_ok=record["labels_ok"],
        mcp_tool_calls=record["mcp_tool_calls"], agent_tool_calls=record["agent_tool_calls"],
        builtin_tool_calls=record["builtin_tool_calls"], mcp_log_matches=record["mcp_log_matches"],
        permission_denials=record["permission_denials"], outputs_written=record["outputs_written"],
        processes_left_in_namespace=record["processes_left_in_namespace"], key_found_in=record["key_found_in"],
        balance_before=record.get("balance_before"), answer=record["answer"])
    manifest["complete"] = manifest_complete(manifest)
    write_json(os.path.join(run_dir, "manifest.json"), manifest)
    return manifest


def summary_row(rec: dict, manifest: dict, note: str) -> dict:
    return dict(run_id=rec["run_id"], task=rec["task"], scenario=rec["scenario"], agent=rec["agent"],
                repeat=rec["repeat"], exit=rec["exit"], seconds=rec["seconds"], completed=rec["completed"],
                model_calls=rec["model_calls"], model_statuses=rec["model_statuses"],
                model_request_bytes=rec["model_request_bytes"], wire_clean=rec["wire_clean"],
                labels_ok=rec["labels_ok"], label_agreement=rec["label_agreement"],
                builtin_tool_calls=rec["builtin_tool_calls"], mcp_tool_calls=len(rec["mcp_tool_calls"]),
                mcp_log_matches=rec["mcp_log_matches"], background_over_limit=rec["background_over_limit"],
                codex_ab_chatgpt_up=rec["codex_ab_chatgpt_up"], manifest_complete=manifest["complete"],
                reran_after_drop=rec.get("reran_after_drop", False), flagged_drop=rec.get("flagged_drop", False),
                drops=len(rec["drops"]), key_found_in=rec["key_found_in"], balance_before=rec.get("balance_before"),
                template=rec["template_home"], note=note, answer=rec["answer"][-60:])


def row_from_manifest(manifest: dict) -> dict:
    """The report row of a run whose manifest was written but whose row was not (a session stopped in between)."""
    m = manifest
    return dict(run_id=m["run_id"], task=m["task"], scenario=m["scenario"], agent=m["agent"], repeat=m["repeat"],
                exit=m["exit"], seconds=m["seconds"], completed=m["completed"], model_calls=m["model_calls"],
                model_statuses=m["model_statuses"], model_request_bytes=m["model_request_bytes"],
                wire_clean=m["wire_verdict"] == "clean", labels_ok=m["labels_ok"],
                label_agreement=m["label_agreement"], builtin_tool_calls=m["builtin_tool_calls"],
                mcp_tool_calls=len(m["mcp_tool_calls"]), mcp_log_matches=m["mcp_log_matches"],
                background_over_limit=m["background_over_limit"], codex_ab_chatgpt_up=m["codex_ab_chatgpt_up"],
                manifest_complete=m["complete"], reran_after_drop=m["reran_after_drop"],
                flagged_drop=m["flagged_drop"], drops=len(m["drops"]), key_found_in=m["key_found_in"],
                balance_before=m["balance_before"], template=m["home_template"], note="row rebuilt from the manifest",
                answer=(m["answer"] or "")[-60:])


def error_row(run_id: str, task: dict, which: str, repeat: int, exc: BaseException) -> dict:
    return dict(run_id=run_id, task=task["id"], scenario=task["scenario"], agent=which, repeat=repeat, error=True,
                note=f"run stopped on {type(exc).__name__}: {exc}",
                traceback=traceback.format_exc()[-2000:])


def describe_run(rec: dict, manifest: dict) -> str:
    return (f"exit {rec['exit']} in {rec['seconds']} s, completed {rec['completed']}; model calls {rec['model_calls']} "
            f"(statuses {rec['model_statuses']}); wire "
            f"{'clean' if rec['wire_clean'] else 'FAIL ' + str(rec['wire_fails'])}; labels ok {rec['labels_ok']} "
            f"(agreement {rec['label_agreement']}); tool calls {rec['agent_tool_calls']}; MCP log matches "
            f"{rec['mcp_log_matches']}; bytes per host {vm_gate.describe_hosts(rec['hosts'])}; manifest complete "
            f"{manifest['complete']}"
            + (f"; background over {BACKGROUND_DOWNLOAD_LIMIT:,} B {rec['background_over_limit']}"
               if rec["background_over_limit"] else "")
            + (f"; API KEY FOUND IN {rec['key_found_in']}" if rec["key_found_in"] else ""))


# ---------------------------------------------------------------- the pilot
def make_templates(gate: vm_gate.Gate) -> dict:
    """One warm template home per agent version, made and checked exactly as the gate's H1 does. The warm-up runs'
    key logs and home copies are deleted once the templates are saved."""
    made = gate.warm_homes()
    templates = {}
    for which in made:
        t = homes.load_template(vm_gate.HOMES_ROOT, which, homes.agent_version(which))
        templates[which] = dict(home=t["home"], id=template_id(t), saved=t.get("saved"),
                                codex_plugins_sha=(t.get("codex_plugins") or {}).get("sha"))
    delete_keylogs(gate.out)
    if os.path.isdir(gate.out):
        for name in os.listdir(gate.out):
            shutil.rmtree(os.path.join(gate.out, name, "home"), ignore_errors=True)
    return templates


def plan(only_task: str | None, only_agent: str | None) -> list[tuple[dict, str, int]]:
    tasks = [t for t in pilot_tasks.TASKS if only_task is None or t["id"] == only_task]
    if only_task and not tasks:
        raise ValueError(f"no task {only_task}")
    agents = [only_agent] if only_agent else AGENTS
    return [(t, a, r) for t in tasks for a in agents for r in range(1, REPEATS + 1)]


def run_name(task: dict, which: str, repeat: int) -> str:
    return f"{task['id']}__{which}__r{repeat}"


def run_done(runs_dir: str, run_id: str) -> bool:
    return os.path.exists(os.path.join(runs_dir, run_id, "manifest.json"))


def set_aside(path: str, suffix: str) -> str:
    """Moves a run folder out of the way (a flagged first attempt, a run that stopped on an error, or a run left by a
    stopped session), without its home copy and key log."""
    shutil.rmtree(os.path.join(path, "home"), ignore_errors=True)
    delete_keylogs(path)
    dest, k = f"{path}__{suffix}1", 1
    while os.path.exists(dest):
        k += 1
        dest = f"{path}__{suffix}{k}"
    os.rename(path, dest)
    return dest


def kept_rows(rows: list) -> list:
    return [r for r in rows if not r.get("flagged_drop") and not r.get("error")]


def quality_marks(rows: list, expected: int, stopped_early: str | None = None, templates_ok: bool = True) -> dict:
    kept = kept_rows(rows)
    pilot_runs = [r for r in rows if not r.get("error")]
    n = len(kept)
    marks = dict(expected_runs=expected, runs=n, error_runs=sum(1 for r in rows if r.get("error")),
                 flagged_first_attempts=sum(1 for r in rows if r.get("flagged_drop")), stopped_early=stopped_early,
                 templates_ok=templates_ok)
    complete_set = n == expected and not stopped_early and templates_ok
    if not n:
        marks.update(Q1_wire=None, Q2_labels=None, Q3_background=None, Q4_manifests=None, Q5_completed=None,
                     complete_set=complete_set, all_pass=False)
        return marks
    clean = sum(1 for r in kept if r["wire_clean"])
    done = sum(1 for r in kept if r["completed"])
    q1 = clean / n >= WIRE_CLEAN_MIN
    q2 = all(r["labels_ok"] is True for r in kept)
    q3 = all(not r["background_over_limit"] for r in pilot_runs)
    q4 = all(r["manifest_complete"] for r in kept)
    q5 = done / n >= COMPLETED_MIN
    marks.update(wire_clean=clean, wire_clean_fraction=round(clean / n, 4), completed=done,
                 completed_fraction=round(done / n, 4), Q1_wire=q1, Q2_labels=q2, Q3_background=q3,
                 Q4_manifests=q4, Q5_completed=q5, complete_set=complete_set)
    marks["all_pass"] = bool(q1 and q2 and q3 and q4 and q5 and complete_set)
    return marks


def codex_background_report(rows: list) -> dict:
    """Codex's upload to ab.chatgpt.com against run time and model calls."""
    out = [dict(run_id=r["run_id"], task=r["task"], seconds=r["seconds"], model_calls=r["model_calls"],
                model_request_bytes=r["model_request_bytes"], ab_chatgpt_up=r["codex_ab_chatgpt_up"],
                up_per_second=round(r["codex_ab_chatgpt_up"] / r["seconds"], 1) if r["seconds"] else None)
           for r in kept_rows(rows) if r["agent"] == "codex"]
    total_s = sum(r["seconds"] for r in out)
    total_up = sum(r["ab_chatgpt_up"] for r in out)
    return dict(runs=out, total_seconds=round(total_s, 1), total_up=total_up,
                up_per_second=round(total_up / total_s, 1) if total_s else None)


def finish(report: dict, out: str, final: bool) -> None:
    report["updated"] = time.strftime("%Y-%m-%dT%H:%M:%S%z")
    report["final"] = final
    report["quality_marks"] = quality_marks(report["runs"], report["expected_runs"], report.get("stopped_early"),
                                            report.get("templates_ok", False))
    report["codex_background"] = codex_background_report(report["runs"])
    write_json(os.path.join(out, "pilot.json"), report)


def run_pilot(key: str, out: str, balance_floor: float, only_task: str | None, only_agent: str | None,
              resume: bool = False) -> dict:
    pilot_tasks.validate()
    runs_dir = os.path.join(out, "runs")
    os.makedirs(runs_dir, exist_ok=True)
    session = time.strftime("%Y%m%d_%H%M%S")
    old = {}
    if resume and os.path.exists(os.path.join(out, "pilot.json")):
        with open(os.path.join(out, "pilot.json")) as f:
            old = json.load(f)
        only_task, only_agent = old.get("only_task"), old.get("only_agent")   # a resumed pilot keeps its plan
    planned = plan(only_task, only_agent)
    report = dict(stamp=os.path.basename(out), started=old.get("started") or time.strftime("%Y-%m-%dT%H:%M:%S%z"),
                  sessions=old.get("sessions", []) + [session], balance_floor=balance_floor,
                  only_task=only_task, only_agent=only_agent, expected_runs=len(planned),
                  balance_before=old.get("balance_before"), runs=old.get("runs", []), stopped_early=None,
                  templates_ok=False, gate=old.get("gate", []))
    gate = vm_gate.Gate(os.path.join(out, "warmups", session), key)

    bal, stop = check_balance(key, balance_floor)
    report["balance_before"] = report["balance_before"] or bal
    print(f"balance before: {bal}", flush=True)
    if stop:
        report["stopped_early"] = f"{stop}, before the warm-up runs"
        finish(report, out, True)
        return report
    if not gate.environment():
        report["stopped_early"] = "V0 environment failed"
        report["gate"] += gate.results
        finish(report, out, True)
        return report
    netns.cleanup_all()
    templates = make_templates(gate)
    report["gate"] += gate.results
    report["templates"] = report.get("templates", []) + [dict(session=session, **templates)]
    agents = sorted({a for _, a, _ in planned})
    report["templates_ok"] = all(a in templates for a in agents)
    finish(report, out, False)

    index = FIRST_INDEX - 1

    def next_index():
        nonlocal index
        index = index + 1 if index < LAST_INDEX else FIRST_INDEX
        return index

    for task, which, repeat in planned:
        run_id = run_name(task, which, repeat)
        run_dir = os.path.join(runs_dir, run_id)
        if run_done(runs_dir, run_id):
            if not any(r["run_id"] == run_id and not r.get("error") for r in report["runs"]):
                with open(os.path.join(run_dir, "manifest.json")) as f:
                    report["runs"].append(row_from_manifest(json.load(f)))
            continue
        if which not in templates:
            print(f"SKIP {run_id}: no warm template home for {which}", flush=True)
            continue
        if os.path.exists(run_dir):
            set_aside(run_dir, "unfinished")   # left by a stopped session: kept, not counted
        bal, stop = check_balance(key, balance_floor)
        if stop:
            report["stopped_early"] = f"{stop}, before {run_id}"
            report["balance_after"] = bal
            print(report["stopped_early"], flush=True)
            finish(report, out, True)
            return report
        print(f"\n=== {run_id} ===", flush=True)
        try:
            rec = run_once(gate, which, task, repeat, next_index(), templates[which], run_dir)
            rec["balance_before"] = bal.get("total_balance")
            note = ""
            if rec["drops"] and rec["has_calls"]:
                note = f"endpoint dropped a model call ({vm_gate.describe_drops(rec['drops'])}); rerun once"
                print(note, flush=True)
                first_dir = set_aside(run_dir, "drop")
                first = dict(rec, run_id=os.path.basename(first_dir), flagged_drop=True)
                first_manifest = write_manifest(first_dir, first)
                report["runs"].append(summary_row(first, first_manifest, "flagged first attempt (endpoint drop)"))
                finish(report, out, False)
                bal, stop = check_balance(key, balance_floor)
                if stop:
                    report["stopped_early"] = f"{stop}, before the rerun of {run_id}"
                    report["balance_after"] = bal
                    finish(report, out, True)
                    return report
                rec = run_once(gate, which, task, repeat, next_index(), templates[which], run_dir)
                rec.update(balance_before=bal.get("total_balance"), reran_after_drop=True)
                if rec["drops"]:
                    note += f"; the rerun had an endpoint drop too ({vm_gate.describe_drops(rec['drops'])})"
            manifest = write_manifest(run_dir, rec)
            print(describe_run(rec, manifest), flush=True)
            report["runs"].append(summary_row(rec, manifest, note))
        except Exception as e:
            row = error_row(run_id, task, which, repeat, e)
            print(f"ERROR {run_id}: {row['note']}", flush=True)
            delete_keylogs(run_dir)
            if os.path.exists(run_dir):
                row["run_dir"] = os.path.basename(set_aside(run_dir, "error"))
            report["runs"].append(row)
            try:
                netns.cleanup_all()
            except Exception:
                pass
        finish(report, out, False)

    report["balance_after"] = read_balance(key)
    finish(report, out, True)
    return report


# ---------------------------------------------------------------- upload bundles to Google Cloud Storage
def upload_blockers(out: str, key: str) -> list[str]:
    """Reasons not to upload: any key log left, or any file that contains the API key."""
    reasons = []
    left = [os.path.relpath(os.path.join(b, f), out) for b, _, fs in os.walk(out) for f in fs if f.endswith(".keys")]
    if left:
        reasons.append(f"key logs still present: {left}")
    hits = homes.find_secret(out, key)
    if hits:
        reasons.append(f"API key found in {hits}")
    return reasons


def upload_to_gcs(out: str, gcs_prefix: str, key: str) -> dict:
    """Copy the pilot output (captures, logs, manifests, labels, pilot.json) to a GCS bucket with gsutil, so a Colab
    cell can bring the bundles into Drive. Refused while any key log is left or any file contains the API key; key
    logs are excluded from the copy as well. gsutil uses the VM's own service-account credentials."""
    dest = gcs_prefix.rstrip("/") + "/" + os.path.basename(out)
    blockers = upload_blockers(out, key)
    if blockers:
        print(f"upload REFUSED: {'; '.join(blockers)}", flush=True)
        return dict(ok=False, dest=dest, refused=blockers)
    r = subprocess.run(["gsutil", "-m", "rsync", "-r", "-x", r".*\.keys$", out, dest], capture_output=True, text=True)
    ok = r.returncode == 0
    print((r.stdout or "")[-2000:]); print((r.stderr or "")[-2000:])
    print(f"{'uploaded to' if ok else 'upload FAILED to'} {dest}", flush=True)
    return dict(ok=ok, dest=dest, returncode=r.returncode)


def stop_on_signal(signum, frame):
    for s in (signal.SIGHUP, signal.SIGTERM):   # a second signal must not cut the clean-up short
        signal.signal(s, signal.SIG_IGN)
    raise SystemExit(f"stopped by signal {signum}")


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--upload", default=None, help="gs://bucket/prefix to copy the finished bundles to (optional)")
    ap.add_argument("--balance-floor", type=float, default=DEFAULT_BALANCE_FLOOR,
                    help="stop before a run if the DeepSeek balance is at or below this")
    ap.add_argument("--only-task", default=None, help="run a single task id (a smoke test)")
    ap.add_argument("--only-agent", default=None, choices=AGENTS, help="run a single agent")
    ap.add_argument("--resume", default=None, metavar="STAMP",
                    help="continue the pilot in data/harness/pilot/STAMP, skipping runs that have a manifest")
    a = ap.parse_args(argv)

    key = os.environ.get("DEEPSEEK_API_KEY", "")
    if not key and sys.stdin.isatty():
        import getpass
        key = getpass.getpass("DeepSeek API key: ")
        os.environ["DEEPSEEK_API_KEY"] = key
    if not key:
        print("DEEPSEEK_API_KEY not set", file=sys.stderr)
        return 1
    for s in (signal.SIGHUP, signal.SIGTERM):
        signal.signal(s, stop_on_signal)
    stale = delete_keylogs(PILOT_ROOT) if os.path.isdir(PILOT_ROOT) else 0
    if a.resume:
        out = os.path.join(PILOT_ROOT, a.resume)
        if not os.path.isdir(out):
            print(f"no pilot to resume at {out}", file=sys.stderr)
            return 1
    else:
        out = os.path.join(PILOT_ROOT, time.strftime("%Y%m%d_%H%M%S"))
        os.makedirs(out, exist_ok=True)
    print(f"benign collection pilot {os.path.basename(out)}{' (resumed)' if a.resume else ''}, output {out}; "
          f"stale key logs deleted: {stale}\n", flush=True)
    report = {}
    try:
        report = run_pilot(key, out, a.balance_floor, a.only_task, a.only_agent, resume=bool(a.resume))
    finally:
        stopped = stop_processes_under(out)
        try:
            netns.cleanup_all()
        except Exception:
            pass
        print(f"\nkey logs deleted: {delete_keylogs(out)}; processes stopped: {stopped}", flush=True)
    marks = report.get("quality_marks", {})
    print("\nPILOT QUALITY MARKS")
    for k in ("Q1_wire", "Q2_labels", "Q3_background", "Q4_manifests", "Q5_completed", "complete_set"):
        print(f"  {'PASS' if marks.get(k) else 'FAIL' if marks.get(k) is not None else 'n/a'}  {k}")
    print(f"\n{'PASS' if marks.get('all_pass') else 'FAIL'}  pilot ({marks.get('runs', 0)} kept runs of "
          f"{marks.get('expected_runs')} planned, wire clean {marks.get('wire_clean_fraction')}, completed "
          f"{marks.get('completed_fraction')}, runs stopped on an error {marks.get('error_runs')})")
    print(f"balance before {(report.get('balance_before') or {}).get('total_balance')}, "
          f"after {(report.get('balance_after') or {}).get('total_balance')}")
    if report.get("stopped_early"):
        print(f"stopped early: {report['stopped_early']}")
    if a.upload and report:
        report["upload"] = upload_to_gcs(out, a.upload, key)
        write_json(os.path.join(out, "pilot.json"), report)
    return 0 if marks.get("all_pass") else 1


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile harness/README.md
# Collection harness (benign runs)

Testbed that runs benign agent tasks and records, for every clean run, the exact byte length of each model call and
each tool call, next to a packet capture of that run. It runs on a Linux VM with root, because packet capture and
per-run network isolation need control of the operating system. Everything after collection runs in Colab.

| File | Role |
|---|---|
| `call_gateway.py` | Gateway the agent calls instead of the model endpoint. Logs the exact request and response body length of every call (JSON lines), forwards the request unchanged apart from Host and the upstream path prefix, streams the response back as it arrives, keeps one upstream connection per agent connection. Upstream TLS keys go to a key log for ground-truth labelling only. `--hold-until` is used only to prepare a warm home. |
| `homes.py` | Warm template homes, one per agent version, and their private copy into each run's home. |
| `mcp_tools.py` | Local MCP tool server (stdio) with benign workspace tools. Logs each tool call's argument size at the tool boundary. |
| `netns.py` | One network namespace per run, with NAT, public DNS, containment rules and a capture on the run's own interface. The veth pair takes the uplink's MTU and has segmentation and receive offloads off, and tcpdump writes every packet as it arrives, so the capture holds wire-size packets and no packet is lost when it stops. |
| `offline_check.py` | Checks that need no network or root: exact sizes from 0 B to 1 MB, unbuffered streaming, connection reuse, reconnects, TLS key log, MCP tool log, and (with `--agents`) Claude Code and Codex against a local fake endpoint. |
| `vm_gate.py` | Checks on the VM before any collection: isolation and capture, DeepSeek Anthropic and Responses APIs through the gateway, key-log decryption, both agents through the gateway, and the traffic of each agent straight to DeepSeek compared with its traffic through the gateway. |
| `pilot_tasks.py` | The ten benign tasks of the collection pilot and their synthetic workspace inputs. |
| `pilot.py` | The collection pilot: 60 benign runs through the harness, per-run labels, tool-call record, cost guard, manifests, resume and the pilot quality marks. |
| `vm_setup.sh` | Installs the system packages (including ethtool), Node.js 22 and the pinned agent versions on Ubuntu 24.04. |

## VM gate

    git clone https://github.com/anasbiswas1/unexplained-traffic.git
    cd unexplained-traffic
    sudo bash harness/vm_setup.sh
    export DEEPSEEK_API_KEY=...        # kept in the environment only, never written to disk
    sudo -E python3 harness/vm_gate.py

Output goes to `data/harness/vm_gate/<time>/` (git-ignored): captures, call logs, tool logs, agent output, a
`net.json` per run (uplink, MTU, offload states, capture counts) and `vm_gate.json`. Key logs are deleted when the gate
ends.

## Warm homes

Benign runs represent a returning user with a warm home, not a first launch. Each agent version has one template home,
made at the start of each gate or collection session by one trivial task run from an empty home and saved under
`data/harness/homes/<agent>-<version>/` with a `manifest.json`. Every run gets its own copy of the template before the
agent starts; templates are never shared between runs. All runs therefore carry the template's installation identity,
as one user's sessions would.

- Codex syncs its plugin catalogue (`github.com/openai/plugins`) in the background at every launch and keeps the result
  only if the sync finishes before the agent exits. From an empty home a short run downloads about 25 MB and throws it
  away, so every such run downloads it again. The warm-up therefore holds its first model call at the gateway until
  the sync has finished (`.codex/.tmp/plugins.sha` exists), and the template is saved only then. A run from that home
  checks the remote commit (`git ls-remote`, about 40 KB) and fetches again only if the catalogue has changed upstream
  since the template was made.
- Claude Code showed no comparable first-launch download in testing (about 14 KB, as on a warm launch); its template
  is made the same way, without the hold.
- No template file may contain the API key.

V6, V7 and V8 report bytes up and down per remote host for every run and fail a run that downloads more than 1,000,000
bytes from any host other than the model host.

## Agent settings on DeepSeek

- Claude Code, from DeepSeek's Claude Code guide: `ANTHROPIC_BASE_URL` (the gateway, which forwards to
  `https://api.deepseek.com/anthropic`), `ANTHROPIC_AUTH_TOKEN`, `ANTHROPIC_MODEL=deepseek-flash[1m]`,
  `ANTHROPIC_DEFAULT_OPUS_MODEL=deepseek-flash[1m]`, `ANTHROPIC_DEFAULT_SONNET_MODEL=deepseek-flash[1m]`,
  `ANTHROPIC_DEFAULT_HAIKU_MODEL=deepseek-flash`. Local MCP tools reach the model as ordinary tool definitions.
- Codex, from DeepSeek's Codex guide: provider `base_url` (the gateway, which forwards to `https://api.deepseek.com`),
  `wire_api = "responses"`, model `deepseek-flash`; the key is read from `DEEPSEEK_API_KEY` through `env_key`.

## Collection pilot

After the gate passes, the pilot collects a small set of benign runs through the harness to measure the call-level
benign floor that ANT could not give (ANT has no per-call sizes). `pilot_tasks.py` holds ten benign tasks across four
of ANT's scenarios (artifact processing 3, business services 2, general knowledge 2, scientific evidence 3);
`pilot.py` runs them with both agents, three times each, one run at a time: 10 tasks x 2 agents x 3 repeats = 60 runs.

Every task is self-contained inside the run's workspace: its inputs are synthetic strings written there before the
agent starts, and no task needs the public internet, a real credential or real personal data. Live web pages are not
used: uncontrolled third-party traffic would break the clean benign floor (model host plus the agent's own background
channel) and differ between the three repeats.

Each agent keeps its default tool set next to the four benign MCP tools. With the settings used here (tested with the
pinned versions), Claude Code in print mode runs its Read tool inside the workspace and read-only Bash commands, and
refuses Read outside the workspace, Write, Edit, WebFetch and WebSearch; Codex (workspace-write sandbox, approval never,
web search disabled) runs shell commands that can read outside the workspace and write inside it, with network blocked
inside its sandbox. The MCP tool log sees only MCP calls, so the pilot also records every tool call, built-in or MCP,
from the agent's own event output (Claude Code `--output-format stream-json`, Codex `exec --json`). Switching to that
output leaves every request byte for byte the same.

Each run keeps, under `data/harness/pilot/<time>/runs/<run_id>/` (git-ignored): the gateway call log, the MCP tool log,
`agent_tools.jsonl` (every tool call from the event output), the capture, `net.json`, the agent output, the workspace,
`labels.json` and `manifest.json`. `labels.json` is the per-call ground-truth table built straight after the run from
its TLS key log, the same decryption the gate's V4 does: each logged call is paired with the decrypted request sent on
the same upstream connection (in order within that connection) and its decrypted length is checked against the
gateway's logged length. The key log is deleted the moment `labels.json` is written and is never a detector input. The
run's home (a copy of the template, about 85 MB for Codex) is deleted after the run; the manifest names the template.

A run whose model call the endpoint drops is rerun once; both records are kept, the first flagged, and the rerun is the
run that counts. A run that stops on an error is set aside and recorded, and the pilot goes on. The cost guard reads
the DeepSeek balance before the warm-up runs and before every run (and rerun), and stops cleanly when the balance
cannot be read, is not available, or is at or below the floor (`--balance-floor`, in the first currency DeepSeek
lists); each run is capped at 300 s, Claude Code also at 12 turns. A stopped pilot is continued with `--resume <time>`,
which skips every run that already has a manifest. `pilot.json` is rewritten after every run.

Pilot quality marks, fixed before the run (kept runs: not a flagged first attempt, not a run that stopped on an error):
Q1 at least 95% of kept runs clean on the wire; Q2 every kept run's labels hold (every model-call request body sent to
the endpoint agrees, every decrypted request is paired, nothing on the model host's connections reads as HTTP without
the key log); Q3 no background download over 1,000,000 bytes from any host other than the model host in any pilot run,
flagged first attempts included; Q4 every kept run's manifest complete; Q5 at least 95% of kept runs completed (exit 0,
every model call answered 200, the agent's own result not an error). The pilot passes only when Q1 to Q5 pass and the
set is complete: every planned run kept, no early stop, a warm template for every agent. Codex's upload to
`ab.chatgpt.com` is reported against run time and model calls.

### Run the pilot on the VM

Run inside `tmux`, so a dropped SSH connection does not stop the pilot (on SIGHUP or SIGTERM the pilot still cleans
up and deletes its key logs):

    tmux new -s pilot
    export DEEPSEEK_API_KEY=...                 # kept in the environment only, never written to disk
    sudo -E python3 harness/pilot.py --only-task artifact_sales_summary --only-agent claude   # smoke test, 3 runs
    sudo -E python3 harness/pilot.py --upload gs://<bucket>/pilot                         # full pilot + upload
    sudo -E python3 harness/pilot.py --resume <time> --upload gs://<bucket>/pilot         # continue a stopped pilot

The upload is refused while any key log is left or any file contains the API key, and key logs are excluded from the
copy.

### Bundles to Drive through a Google Cloud Storage bucket

The analysis runs in Colab, so the run bundles must reach the repository's git-ignored `data/` folder on Drive
(`unexplained-traffic/data/harness/pilot/<time>/`, next to `data/ant/`). The VM uploads to a bucket in the same
project, then a Colab cell copies the bundles into Drive. No credentials are written to the repo; the VM uses its own
service-account identity and Colab uses the signed-in Google account. One-time setup, run in Cloud Shell or on the VM
(set `VM` to the VM's name and `BUCKET` to a new, globally unique bucket name):

    VM=ut-harness
    BUCKET=<bucket>
    PROJECT=$(gcloud config get-value project)
    ZONE=$(gcloud compute instances list --filter="name=$VM" --format="value(zone.basename())")
    REGION=${ZONE%-*}
    SA=$(gcloud compute instances describe $VM --zone $ZONE --format="value(serviceAccounts[0].email)")
    echo '{"rule": [{"action": {"type": "Delete"}, "condition": {"age": 30}}]}' > lifecycle.json
    gcloud storage buckets create gs://$BUCKET --project $PROJECT --location $REGION \
        --uniform-bucket-level-access --lifecycle-file lifecycle.json
    gcloud storage buckets add-iam-policy-binding gs://$BUCKET --member "serviceAccount:$SA" \
        --role roles/storage.objectAdmin

The lifecycle rule deletes objects 30 days after upload; the Drive copy is the one that stays. Then check on the VM
that it can write to the bucket:

    gsutil ls gs://<bucket>
    echo ok | gsutil cp - gs://<bucket>/_writetest && gsutil rm gs://<bucket>/_writetest

If this fails with an access-scope error, the VM was created without a storage write scope. Add it, keeping the
default scopes (the VM must be stopped; `--scopes` replaces the VM's scope list):

    gcloud compute instances stop $VM --zone $ZONE
    gcloud compute instances set-service-account $VM --zone $ZONE --scopes default,storage-rw
    gcloud compute instances start $VM --zone $ZONE

After the pilot, the notebook's Drive cell copies `gs://<bucket>/pilot/<time>` into
`data/harness/pilot/<time>` in the repository folder on Drive.

## Fixed rules

- Benign tasks only. No real credentials and no real user data in any run.
- The decrypted view of a capture exists only to label which bytes were real content, as a check on the byte-budget
  analysis. It is never an input to the detector.
- Key logs are deleted with the testbed and never released. Captures, key logs and call logs live under `data/` and
  are never committed. Run bundles move from the VM to Drive through a Google Cloud Storage bucket, with key logs
  already deleted; no credentials go in the repo.

In [ ]:
import py_compile, glob
for f_ in sorted(glob.glob('harness/*.py')):
    py_compile.compile(f_, doraise=True)
for f_ in ('pilot_tasks.py', 'pilot.py', 'README.md'):
    print(f'{f_:16s} {os.path.getsize(f"harness/{f_}"):>8,d} bytes')
print('compiled', len(glob.glob('harness/*.py')), 'harness modules')

## Stage B. Offline checks

The pilot logic that needs no capture (P1 to P13), then the existing harness offline checks (O1 to O14) to confirm
nothing in the harness regressed.

In [ ]:
import importlib, json, os, sys, tempfile
sys.path.insert(0, 'harness')
import vm_gate, pilot_tasks, pilot
importlib.reload(vm_gate); importlib.reload(pilot_tasks); importlib.reload(pilot)
PCHK = []


def chk(name, ok, detail):
    PCHK.append((name, bool(ok), detail))


# P1 task set: ten tasks, unique ids, spread 3/2/2/3 by scenario, no path outside the workspace; a moved task is caught
try:
    pilot_tasks.validate()
    spread = {s: len(v) for s, v in pilot_tasks.by_scenario().items()}
    moved = pilot_tasks.TASKS[0]['scenario']
    pilot_tasks.TASKS[0]['scenario'] = 'business services'
    try:
        pilot_tasks.validate()
        caught = False
    except ValueError:
        caught = True
    finally:
        pilot_tasks.TASKS[0]['scenario'] = moved
    chk('P1 task set well formed', spread == pilot_tasks.SPREAD and len(pilot_tasks.TASKS) == 10 and caught,
        dict(tasks=len(pilot_tasks.TASKS), spread=spread, moved_task_caught=caught))
except Exception as e:
    chk('P1 task set well formed', False, f'{type(e).__name__}: {e}')

# P2-P6 labels. tshark is replaced by fixed rows (frame, stream, client port, method, uri, declared length) so the
# pairing and the checks run with no capture.
_orig = vm_gate.tshark_rows
SCN = dict(req=[], plain=[])
vm_gate.tshark_rows = lambda pcap, flt, fields, keylog=None: SCN['req'] if flt.startswith('http.request') else SCN['plain']


def call(conn, seq, port, n, t, path='/v1/messages', framing='length'):
    return dict(conn=conn, seq=seq, up_conn=conn, up_local=f'10.200.51.2:{port}' if port else None, method='POST',
                path=path, req_framing=framing, req_body_bytes=n, t=t, status=200)


one = [call(1, 1, 40001, 1200, 1.0), call(1, 2, 40001, 50, 2.0, '/v1/messages?beta=true'), call(1, 3, 40001, 900, 3.0)]
SCN['req'] = [['10', '0', '40001', 'POST', '/anthropic/v1/messages', '1200'],
              ['20', '0', '40001', 'POST', '/anthropic/v1/messages', '50'],
              ['30', '0', '40001', 'POST', '/anthropic/v1/messages', '900']]
lab = pilot.label_calls('x.pcap', 'k.keys', one, ['0'])
chk('P2 labels: clean run, all model calls agree', lab['ok'] and lab['agreement'] == 1.0 and lab['model_pairs'] == 3,
    dict(agreement=lab['agreement'], model_pairs=lab['model_pairs'], ok=lab['ok']))

SCN['req'][1][5] = '49'   # logged 50, decrypted 49
lab = pilot.label_calls('x.pcap', 'k.keys', one, ['0'])
bad = [p for p in lab['pairs'] if not p['agree']]
chk('P3 labels: one mismatch caught', not lab['ok'] and abs(lab['agreement'] - 2 / 3) < 1e-9 and len(bad) == 1
    and bad[0]['seq'] == 2 and bad[0]['decrypted'] == 49 and bad[0]['logged'] == 50,
    dict(agreement=round(lab['agreement'], 3), mismatch=[(p['seq'], p['logged'], p['decrypted']) for p in bad]))

# P4: two connections; the first call is slow, so the log (written when a call ends) is in a different order from
# the wire. Pairing by connection keeps every pair right.
logged = [call(2, 1, 40003, 500, 1.3), call(2, 2, 40003, 300, 1.5), call(1, 1, 40002, 1200, 1.0)]
SCN['req'] = [['10', '0', '40002', 'POST', '/v1/messages', '1200'],
              ['14', '1', '40003', 'POST', '/v1/messages', '500'],
              ['18', '1', '40003', 'POST', '/v1/messages', '300']]
lab = pilot.label_calls('x.pcap', 'k.keys', logged, ['0', '1'])
chk('P4 labels: overlapping calls on two connections paired by connection', lab['ok'] and lab['agreement'] == 1.0,
    dict(pairs=[(p['logged'], p['decrypted']) for p in lab['pairs']], agreement=lab['agreement']))

# P5: a call whose upstream connect failed never reached the wire: listed, not counted, and no other pair shifts
failed = dict(call(3, 1, None, 0, 1.1), req_framing=None, status=None, error='upstream connect failed')
lab = pilot.label_calls('x.pcap', 'k.keys', [failed] + logged, ['0', '1'])
chk('P5 labels: a call that never reached the endpoint is listed, not counted',
    lab['ok'] and lab['model_pairs'] == 3 and len(lab['not_on_wire']) == 1,
    dict(model_pairs=lab['model_pairs'], not_on_wire=len(lab['not_on_wire']), agreement=lab['agreement']))

# P6: labels fail on a decrypted request with no logged call, on HTTP readable without the key log, and on a frame
# holding two requests
SCN['req'] = SCN['req'] + [['22', '1', '40003', 'POST', '/v1/messages', '77']]
extra = pilot.label_calls('x.pcap', 'k.keys', logged, ['0', '1'])
SCN['req'] = SCN['req'][:3]
SCN['plain'] = [['5']]
plain = pilot.label_calls('x.pcap', 'k.keys', logged, ['0', '1'])
SCN['plain'] = []
SCN['req'] = SCN['req'] + [['24', '1', '40003', 'HEAD,POST', '/a,/b', '']]
amb = pilot.label_calls('x.pcap', 'k.keys', logged, ['0', '1'])
vm_gate.tshark_rows = _orig
chk('P6 labels: unpaired request, plain HTTP and two-request frames fail',
    not extra['ok'] and len(extra['unpaired_decrypted']) == 1 and not plain['ok'] and plain['without_keylog'] == 1
    and not amb['ok'] and amb['ambiguous_frames'] == ['24'],
    dict(unpaired=len(extra['unpaired_decrypted']), without_keylog=plain['without_keylog'],
         ambiguous=amb['ambiguous_frames']))

# P7 tool calls from the agents' event output (shapes as Claude Code 2.1.302 and Codex 0.163.0 print them)
cc = '\n'.join(json.dumps(x) for x in [
    {'type': 'system', 'subtype': 'init', 'tools': ['Read', 'mcp__bench__read_file']},
    {'type': 'assistant', 'parent_tool_use_id': None, 'message': {'content': [
        {'type': 'tool_use', 'id': 't1', 'name': 'Read', 'input': {'file_path': '/w/data/sales.csv'}},
        {'type': 'tool_use', 'id': 't2', 'name': 'mcp__bench__write_file', 'input': {'path': 'out/a.txt',
                                                                                   'content': 'é'}}]}},
    {'type': 'user', 'message': {'content': [
        {'type': 'tool_result', 'tool_use_id': 't1', 'content': 'abc'},
        {'type': 'tool_result', 'tool_use_id': 't2', 'content': [{'type': 'text', 'text': 'ok'}], 'is_error': False}]}},
    {'type': 'assistant', 'parent_tool_use_id': 't9', 'message': {'content': [
        {'type': 'tool_use', 'id': 't3', 'name': 'Bash', 'input': {'command': 'ls'}}]}},
    {'type': 'user', 'message': {'content': [{'type': 'tool_result', 'tool_use_id': 't3', 'content': 'denied',
                                              'is_error': True}]}},
    {'type': 'result', 'subtype': 'success', 'is_error': False, 'result': 'A-100', 'num_turns': 3,
     'permission_denials': [{'tool_name': 'Bash'}]}])
ce = pilot.claude_events('not json\n' + cc)
cx = '\n'.join(json.dumps(x) for x in [
    {'type': 'thread.started'},
    {'type': 'item.completed', 'item': {'id': 'i0', 'type': 'error', 'message': 'model metadata not found'}},
    {'type': 'item.completed', 'item': {'id': 'i1', 'type': 'command_execution', 'command': "/bin/bash -lc 'cat x'",
                                        'aggregated_output': 'hello\n', 'exit_code': 0, 'status': 'completed'}},
    {'type': 'item.completed', 'item': {'id': 'i2', 'type': 'mcp_tool_call', 'server': 'bench', 'tool': 'write_file',
                                        'arguments': {'path': 'out/a.txt', 'content': 'x'},
                                        'result': {'content': [{'type': 'text', 'text': 'written'}]},
                                        'error': None, 'status': 'completed'}},
    {'type': 'item.completed', 'item': {'id': 'i3', 'type': 'file_change', 'changes': [], 'status': 'completed'}},
    {'type': 'item.completed', 'item': {'id': 'i4', 'type': 'agent_message', 'text': 'done'}},
    {'type': 'turn.failed', 'error': {'message': 'stream ended'}}])
xe = pilot.codex_events(cx)
cc_ok = ([(c['tool'], c['source'], c['input_bytes'], c['output_bytes'], c['parent']) for c in ce['tool_calls']] ==
         [('Read', 'builtin', 33, 3, None), ('mcp__bench__write_file', 'mcp', 35, 2, None),
          ('Bash', 'builtin', 16, 6, 't9')]
         and ce['answer'] == 'A-100' and ce['error'] is None and ce['permission_denials'] == 1)
cx_ok = ([(c['tool'], c['source'], c['input_bytes'], c['output_bytes'], c['is_error']) for c in xe['tool_calls']] ==
         [('command_execution', 'builtin', 21, 6, False), ('mcp__bench__write_file', 'mcp', 34, 7, False),
          ('file_change', 'builtin', None, None, False)] and xe['error'] == 'stream ended')
no_result = pilot.claude_events(cc.rsplit('\n', 1)[0])
chk('P7 agent tool calls from event output', cc_ok and cx_ok and no_result['error'] == 'no result event',
    dict(claude=ce['tool_calls'], codex=xe['tool_calls'], codex_error=xe['error']))

# P8 manifest completeness: exit 0 and an empty status list count as present; an empty host table, a missing field
# or missing labels.json fail
good = dict(run_id='a', task='t', scenario='s', agent='claude', version='2.1.302', model='deepseek-flash', repeat=1,
            home_template='claude-2.1.302@x', start='s', end='e', exit=0, model_statuses=[200], wire_verdict='clean',
            bytes_per_host={'h': {}}, labels_written=True)
cases = dict(good=good, empty_hosts=dict(good, bytes_per_host={}), missing_exit=dict(good, exit=None),
             no_labels=dict(good, labels_written=False), no_template=dict(good, home_template=''))
res = {k: pilot.manifest_complete(v) for k, v in cases.items()}
chk('P8 manifest completeness', res == dict(good=True, empty_hosts=False, missing_exit=False, no_labels=False,
                                            no_template=False), res)


# P9 quality marks
def _row(**kw):
    b = dict(run_id='r', agent='claude', task='t', seconds=10.0, model_calls=2, model_request_bytes=1000,
             codex_ab_chatgpt_up=0, wire_clean=True, labels_ok=True, background_over_limit=[], manifest_complete=True,
             completed=True, flagged_drop=False)
    b.update(kw)
    return b


full = [_row() for _ in range(20)]
m = {
    'all good': pilot.quality_marks(full, 20),
    'wire 19/20': pilot.quality_marks(full[:19] + [_row(wire_clean=False)], 20),
    'wire 18/20': pilot.quality_marks(full[:18] + [_row(wire_clean=False)] * 2, 20),
    'labels one bad': pilot.quality_marks(full[:19] + [_row(labels_ok=False)], 20),
    'background in flagged attempt': pilot.quality_marks(
        full + [_row(flagged_drop=True, background_over_limit=[('github.com', 2_000_000)])], 20),
    'completed 18/20': pilot.quality_marks(full[:18] + [_row(completed=False)] * 2, 20),
    'one run missing': pilot.quality_marks(full[:19], 20),
    'stopped early': pilot.quality_marks(full, 20, stopped_early='balance at floor'),
    'no template': pilot.quality_marks(full, 20, templates_ok=False),
    'flagged and error rows excluded': pilot.quality_marks(
        full + [_row(flagged_drop=True, wire_clean=False), dict(run_id='e', error=True)], 20)}
want = {'all good': True, 'wire 19/20': True, 'wire 18/20': False, 'labels one bad': False,
        'background in flagged attempt': False, 'completed 18/20': False, 'one run missing': False,
        'stopped early': False, 'no template': False, 'flagged and error rows excluded': True}
got = {k: v['all_pass'] for k, v in m.items()}
chk('P9 quality marks and complete set', got == want and m['flagged and error rows excluded']['runs'] == 20
    and not m['wire 18/20']['Q1_wire'] and not m['completed 18/20']['Q5_completed']
    and not m['background in flagged attempt']['Q3_background'], got)

# P10 cost guard: the reply is parsed, every stop reason is found, and a failed read returns an error without raising
# (the failed read goes to a closed local port, so nothing leaves this machine)
ok_body = {'is_available': True, 'balance_infos': [{'currency': 'USD', 'total_balance': '12.50'}]}
parsed = pilot.parse_balance(ok_body)
reasons = dict(
    ok=pilot.balance_stop(parsed, 1.0),
    at_floor=pilot.balance_stop(pilot.parse_balance({'is_available': True, 'balance_infos': [
        {'currency': 'USD', 'total_balance': '1.00'}]}), 1.0),
    not_available=pilot.balance_stop(dict(parsed, is_available=False), 1.0),
    no_total=pilot.balance_stop(pilot.parse_balance({'is_available': True, 'balance_infos': []}), 1.0))
failed_read = pilot.read_balance('test-key', url='http://127.0.0.1:9/user/balance', timeout=5)
reasons['read_failed'] = pilot.balance_stop(failed_read, 1.0)
chk('P10 cost guard', parsed['total_balance'] == 12.5 and reasons['ok'] is None
    and all(reasons[k] for k in ('at_floor', 'not_available', 'no_total', 'read_failed')), reasons)

# P11 upload refusals: a key log left, or a file holding the API key, blocks the upload; a clean folder does not
with tempfile.TemporaryDirectory() as d:
    os.makedirs(os.path.join(d, 'runs', 'a'))
    with open(os.path.join(d, 'runs', 'a', 'manifest.json'), 'w') as f:
        f.write('{}')
    clean = pilot.upload_blockers(d, 'test-key-123')
    with open(os.path.join(d, 'runs', 'a', 'gw.keys'), 'w') as f:
        f.write('CLIENT_RANDOM x y\n')
    with open(os.path.join(d, 'runs', 'a', 'agent_stdout.txt'), 'w') as f:
        f.write('token test-key-123')
    blocked = pilot.upload_blockers(d, 'test-key-123')
    swept = pilot.delete_keylogs(d)
chk('P11 upload refused on a key log or the API key', clean == [] and len(blocked) == 2 and swept == 1, blocked)

# P12 resume: a run counts as done only with a manifest; a folder left behind is set aside, not reused, without its
# home copy and key log; a row rebuilt from a manifest scores the same as the row written with it
with tempfile.TemporaryDirectory() as d:
    rd = os.path.join(d, 'runs')
    os.makedirs(os.path.join(rd, 'x__claude__r1'))
    before = pilot.run_done(rd, 'x__claude__r1')
    moved = os.path.basename(pilot.set_aside(os.path.join(rd, 'x__claude__r1'), 'unfinished'))
    os.makedirs(os.path.join(rd, 'x__claude__r1', 'home', '.codex'))
    with open(os.path.join(rd, 'x__claude__r1', 'gw.keys'), 'w') as f:
        f.write('CLIENT_RANDOM x y\n')
    moved2_path = pilot.set_aside(os.path.join(rd, 'x__claude__r1'), 'unfinished')
    moved2 = os.path.basename(moved2_path)
    stripped = sorted(os.listdir(moved2_path)) == []
    rec = dict(run_id='x__claude__r1', task='x', scenario='s', agent='claude', version='2.1.302',
               model='deepseek-flash', repeat=1, template_home='claude-2.1.302@t', template_saved='t',
               codex_plugins_sha=None, start='s', end='e', seconds=12.0, exit=0, model_calls=2,
               model_statuses=[200, 200], model_request_bytes=150000, endpoint_errors=[], agent_error=None,
               completed=True, wire_clean=True, wire_fails=[], hosts={'api.deepseek.com': {'conns': 1, 'up': 1,
                                                                                            'down': 1}},
               drops=[], background_over_limit=[], codex_ab_chatgpt_up=0, labels_written=True, label_agreement=1.0,
               labels_ok=True, mcp_tool_calls=[('read_file', 30)], agent_tool_calls=[], builtin_tool_calls=0,
               mcp_log_matches=True, permission_denials=0, outputs_written=[], processes_left_in_namespace=0,
               key_found_in=[], answer='A-100')
    os.makedirs(os.path.join(rd, 'x__claude__r1'))
    man = pilot.write_manifest(os.path.join(rd, 'x__claude__r1'), rec)
    after = pilot.run_done(rd, 'x__claude__r1')
    row_a = pilot.summary_row(rec, man, '')
    with open(os.path.join(rd, 'x__claude__r1', 'manifest.json')) as f:
        row_b = pilot.row_from_manifest(json.load(f))
same = pilot.quality_marks([row_a], 1) == pilot.quality_marks([row_b], 1)
chk('P12 resume helpers', not before and after and moved == 'x__claude__r1__unfinished1'
    and moved2 == 'x__claude__r1__unfinished2' and stripped and man['complete'] and same,
    dict(done_before=before, done_after=after, set_aside=[moved, moved2], home_and_key_log_removed=stripped,
         same_marks=same))

# P13 a process left behind by a stopped run (its command line names a file in the pilot folder) is stopped at the end
import subprocess, time
with tempfile.TemporaryDirectory() as d:
    stray = subprocess.Popen(['python3', '-c', 'import time; time.sleep(60)', os.path.join(d, 'runs', 'x', 'gw.keys')])
    other = subprocess.Popen(['python3', '-c', 'import time; time.sleep(60)'])
    time.sleep(0.5)
    n = pilot.stop_processes_under(d)
    time.sleep(0.5)
    stray_gone, other_alive = stray.poll() is not None, other.poll() is None
    other.kill(); other.wait(); stray.wait()
chk('P13 stray processes of the pilot stopped, others left alone', n == 1 and stray_gone and other_alive,
    dict(stopped=n, stray_gone=stray_gone, other_alive=other_alive))

for name, ok, val in PCHK:
    print(f"{'PASS' if ok else 'FAIL'}  {name}: {val}")
PILOT_PASS = all(ok for _, ok, _ in PCHK)
print('\nP-series:', 'PASS' if PILOT_PASS else 'FAIL')

In [ ]:
import json, subprocess
OUT = 'data/harness/offline_02e'
r = subprocess.run([sys.executable, 'harness/offline_check.py', '--out', OUT], capture_output=True, text=True,
                   timeout=900)
print(r.stdout[-6000:]); print(r.stderr[-2000:])
with open(f'{OUT}/offline_check.json') as f:
    OFFLINE = json.load(f)
print('\nexisting harness offline checks:',
      'PASS' if all(x['passed'] is True for x in OFFLINE) else
      ('has skips' if any(x['passed'] is None for x in OFFLINE) else 'FAIL'))

## Stage C. Decision

Records the Colab-side result (P1 to P13 and the existing offline checks) to `reports/harness/decision_02e.txt` and
`reports/harness/offline_check_02e.json`, and appends to the results log. The pilot's own pass or fail (Q1 to Q5 and
the complete set) is produced on the VM.

In [ ]:
import pandas as pd
import reslog
RES = [dict(check=c, where='offline', passed=bool(ok), detail=str(v)) for c, ok, v in PCHK]
RES += [dict(check=x['check'], where='offline', passed=x['passed'], detail=x['detail']) for x in OFFLINE]
TAB = pd.DataFrame(RES)[['check', 'where', 'passed', 'detail']]
TAB.to_json('reports/harness/offline_check_02e.json', orient='records', indent=1)
DECISION = 'PASS' if (TAB.passed == True).all() else 'FAIL'
mark = lambda p: 'PASS' if p is True else ('SKIP' if p is None else 'FAIL')
TEXT = ('DECISION: ' + DECISION + ' (Colab-side checks: pilot logic P1-P13 and the existing harness offline checks '
        'O1-O14. The 60 benign runs, captures, wire verdict, per-run key-log labels, bytes per host and the pilot '
        'quality marks Q1-Q5 run on the VM in harness/pilot.py.)\n\n'
        + '\n'.join(f'{mark(p)}  {c}: {d}' for c, p, d in zip(TAB.check, TAB.passed, TAB.detail)))
with open('reports/harness/decision_02e.txt', 'w') as f:
    f.write(TEXT + '\n')
reslog.log_result('02e benign collection pilot: Colab-side checks', TEXT)
pd.set_option('display.max_colwidth', 400)
display(TAB)
assert DECISION == 'PASS', 'a Colab-side check failed: see the table'

## Stage D. Run the pilot on the VM

These commands run in Cloud Shell and on the VM, not in Colab; the cell below only prints them.

In [ ]:
print(r"""
# ---- one-time GCS setup (Cloud Shell or the VM); VM is the VM's name, BUCKET a new globally unique name ----
VM=ut-harness
BUCKET=<bucket>
PROJECT=$(gcloud config get-value project)
ZONE=$(gcloud compute instances list --filter="name=$VM" --format="value(zone.basename())")
REGION=${ZONE%-*}
SA=$(gcloud compute instances describe $VM --zone $ZONE --format="value(serviceAccounts[0].email)")
echo '{"rule": [{"action": {"type": "Delete"}, "condition": {"age": 30}}]}' > lifecycle.json
gcloud storage buckets create gs://$BUCKET --project $PROJECT --location $REGION --uniform-bucket-level-access --lifecycle-file lifecycle.json
gcloud storage buckets add-iam-policy-binding gs://$BUCKET --member "serviceAccount:$SA" --role roles/storage.objectAdmin
# on the VM: check it can write to the bucket
gsutil ls gs://<bucket>
echo ok | gsutil cp - gs://<bucket>/_writetest && gsutil rm gs://<bucket>/_writetest
# only if that fails with an access-scope error: add the storage write scope, keeping the default scopes
gcloud compute instances stop $VM --zone $ZONE
gcloud compute instances set-service-account $VM --zone $ZONE --scopes default,storage-rw
gcloud compute instances start $VM --zone $ZONE

# ---- run the pilot (on the VM, from the repo root, inside tmux) ----
tmux new -s pilot
cd unexplained-traffic && git pull
sudo bash harness/vm_setup.sh                                  # if the VM is fresh
export DEEPSEEK_API_KEY=...                                    # environment only, never written to disk
sudo -E python3 harness/vm_gate.py                             # gate first (unchanged)
sudo -E python3 harness/pilot.py --only-task artifact_sales_summary --only-agent claude    # smoke test, 3 runs
sudo -E python3 harness/pilot.py --upload gs://<bucket>/pilot                          # full pilot + upload
sudo -E python3 harness/pilot.py --resume <stamp> --upload gs://<bucket>/pilot         # only if it stopped
# output: data/harness/pilot/<stamp>/ with runs/<run_id>/ bundles and pilot.json; key logs deleted as it goes
""")

## Stage E. Bring the finished bundles into Drive

After the VM run, copies that run's bundles from the bucket into `data/harness/pilot/<stamp>` in the repository folder
on Drive (git-ignored, next to `data/ant`). Set `BUCKET` and `STAMP` first.

In [ ]:
import os, json, subprocess
BUCKET = 'gs://<bucket>/pilot'
STAMP  = '<stamp>'     # the folder under data/harness/pilot/ on the VM, e.g. 20261024_013000

if '<' in BUCKET or '<' in STAMP:
    print('Set BUCKET and STAMP first, then re-run this cell.')
else:
    from google.colab import auth
    auth.authenticate_user()
    os.chdir(REPO_DIR)
    dest = f'data/harness/pilot/{STAMP}'
    os.makedirs(dest, exist_ok=True)
    src = f'{BUCKET}/{STAMP}'
    r = subprocess.run(['gsutil', '-m', 'rsync', '-r', '-x', r'.*\.keys$', src, dest], capture_output=True, text=True)
    print(r.stdout[-3000:]); print(r.stderr[-3000:])
    print('copied' if r.returncode == 0 else 'copy FAILED', src, '->', dest)
    left = [os.path.join(b, f) for b, _, fs in os.walk(dest) for f in fs if f.endswith('.keys')]
    print('key logs in the copy:', left or 'none')
    if os.path.exists(f'{dest}/pilot.json'):
        with open(f'{dest}/pilot.json') as f:
            rep = json.load(f)
        print('final report:', rep.get('final'), '| stopped early:', rep.get('stopped_early'))
        print('pilot quality marks:', rep.get('quality_marks'))
        print('runs recorded:', len(rep.get('runs', [])))

## Stage F. Commit

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
assert branch == "main", f"the repository is on branch {branch!r}, not main: switch to main before committing"
r = subprocess.run(["python", "tools/commit_cell.py", 'nb02e: benign collection pilot on the harness. pilot_tasks.py: ten benign tasks across four ANT scenarios (spread 3/2/2/3, checked per scenario), synthetic workspace inputs only, no live web. pilot.py: 10 tasks x 2 agents x 3 repeats = 60 runs through the gateway from warm template homes, one run per namespace at a time, reusing the gate building blocks; agents keep their default tool sets and every tool call is recorded from their event output (stream-json, exec --json); per-run labels.json from the key log, calls paired by upstream connection, key log deleted straight after; cost guard on the DeepSeek balance before warm-ups, runs and reruns; per-run error handling, pilot.json after every run, SIGHUP/SIGTERM cleanup (key logs, home copies, namespaces and processes of an interrupted run), --resume; run homes deleted, template id in the manifest; upload refused on a key log or the API key; quality marks Q1-Q5 and the complete-set rule; Codex ab.chatgpt.com upload against run time and model calls. README collection-pilot section with tmux, resume and the GCS setup (lifecycle rule, scopes kept). Colab-side checks P1-P13 and existing offline checks O1-O14; reports/harness/offline_check_02e.json, decision_02e.txt'], capture_output=True, text=True)
print(r.stdout); print(r.stderr)